# OilWatch — train the oil-slick model on Colab (Run All)

**What this does, end to end, on real data:**
1. mounts Google Drive (everything important is saved there → safe against disconnects)
2. installs the `oilspill` package (source embedded below — no GitHub needed)
3. checks the GPU
4. builds the dataset: Cerulean public slick polygons (labels) + Sentinel-1 GRD VV from Microsoft Planetary Computer, warped onto Cerulean's grid
5. overfit sanity gate → full training (U-Net ResNet34, AMP, resumable)
6. ONNX export + honest evaluation (human-reviewed test scenes only)
7. writes `MyDrive/oilspill/oilspill_model.zip` → download it and put `model.onnx` + `model_card.json` into the project's `models/` folder

**Runtime → Change runtime type → T4 GPU** first. Then **Run All**.
If Colab disconnects: reconnect and Run All again — every step resumes from Drive.

Data attribution: SkyTruth Cerulean (cerulean.skytruth.org), CC BY-SA 4.0 (non-commercial conservation use per SkyTruth ToS). Contains modified Copernicus Sentinel data 2023–2026.

In [ ]:
# 1) Google Drive (persistent storage)
import os, pathlib
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = pathlib.Path('/content/drive/MyDrive/oilspill')
except Exception as e:
    print('!! Drive mount failed:', e)
    print('!! Falling back to /content/oilspill_out (NOT persistent; download the zip at the end)')
    DRIVE = pathlib.Path('/content/oilspill_out')
DRIVE.mkdir(parents=True, exist_ok=True)
RUN = DRIVE / 'run1'
DATA = pathlib.Path('/content/data/dataset')
os.environ['OILSPILL_DATA_DIR'] = '/content/data'
print('persistent dir:', DRIVE)

In [ ]:
FILES = {"src/oilspill/__init__.py": "\"\"\"Open-source reproduction of SkyTruth Cerulean's Sentinel-1 oil-slick detection pipeline.\"\"\"\n\n__version__ = \"0.1.0\"\n", "src/oilspill/config.py": "\"\"\"Single source of truth for grid constants, class maps, thresholds and paths.\n\nNothing in this package may hardcode a grid constant or threshold; import it\nfrom here. Values tagged [SRC] are reproduced from Cerulean's published\ninference behaviour; [DERIVED] values are computed from those; [OURS] are our\nown choices and are recorded in DECISIONS.md.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nfrom dotenv import load_dotenv\n\nload_dotenv()\n\n# ---------------------------------------------------------------------------\n# Paths\n# ---------------------------------------------------------------------------\n\nREPO_ROOT = Path(__file__).resolve().parents[2]\n\n\ndef _env_dir(var: str, default: Path) -> Path:\n    raw = os.getenv(var, \"\").strip()\n    return Path(raw).expanduser() if raw else default\n\n\n#: Root for bulky derived artifacts. Defaults to <repo>/data but should point\n#: outside any cloud-synced folder (see .env.example).\nDATA_DIR = _env_dir(\"OILSPILL_DATA_DIR\", REPO_ROOT / \"data\")\nMODELS_DIR = _env_dir(\"OILSPILL_MODELS_DIR\", REPO_ROOT / \"models\")\nREPORTS_DIR = _env_dir(\"OILSPILL_REPORTS_DIR\", REPO_ROOT / \"reports\")\n\nCACHE_DIR = DATA_DIR / \"cache\"\nCERULEAN_CACHE_DIR = CACHE_DIR / \"cerulean\"\nSTAC_CACHE_DIR = CACHE_DIR / \"stac\"\nSCENE_CACHE_DIR = CACHE_DIR / \"scenes\"\nTILES_DIR = DATA_DIR / \"tiles\"\nMASKS_DIR = DATA_DIR / \"masks\"\nSTATUS_DIR = DATA_DIR / \"status\"\nSPLITS_DIR = REPO_ROOT / \"data\" / \"splits\"  # small, tracked in git\nCHECKPOINTS_DIR = DATA_DIR / \"checkpoints\"\n\n\ndef ensure_dirs() -> None:\n    \"\"\"Create every directory this package writes to. Idempotent.\"\"\"\n    for d in (\n        DATA_DIR, MODELS_DIR, REPORTS_DIR, CACHE_DIR, CERULEAN_CACHE_DIR,\n        STAC_CACHE_DIR, SCENE_CACHE_DIR, TILES_DIR, MASKS_DIR, STATUS_DIR,\n        SPLITS_DIR, CHECKPOINTS_DIR,\n    ):\n        d.mkdir(parents=True, exist_ok=True)\n\n\n# ---------------------------------------------------------------------------\n# Grid - morecantile WorldCRS84Quad, zoom 9, tile scale 2  [SRC]\n# ---------------------------------------------------------------------------\n\nGRID_ZOOM = 9\nGRID_TILE_PX = 512\n#: EPSG:4326 degrees per pixel: 180 / 2**9 / 512  [DERIVED from SRC]\nPIXEL_DEG = 180.0 / (2**GRID_ZOOM) / GRID_TILE_PX  # 0.0006866455078125\n#: Grid origin. WorldCRS84Quad is anchored at the antimeridian / north pole.\nGRID_ORIGIN_LON = -180.0\nGRID_ORIGIN_LAT = 90.0\n#: Approximate ground sample distance at the equator, metres. Cerulean's docs\n#: say ~80 m and tile_width_m 40844; PIXEL_DEG * 111320 gives ~76.4 m. The\n#: discrepancy is theirs; we use the exact degree value throughout.\nPIXEL_M_EQUATOR = PIXEL_DEG * 111_320.0\n\nTILE_PX = 512\n#: Sliding-window stride for inference; overlap = TILE_PX - INFER_STRIDE = 128.\nINFER_STRIDE = 384\n\nNODATA = 0\n#: EPSG:4326 is lon/lat; every raster this package writes uses this CRS.\nCRS_EPSG = 4326\nCRS = \"EPSG:4326\"\n\n\n# ---------------------------------------------------------------------------\n# Radiometry\n# ---------------------------------------------------------------------------\n\ndef dn_to_db(dn):\n    \"\"\"Raw Sentinel-1 GRD digital number -> pseudo-dB.\n\n    ``10*log10(DN**2 + 1)``. The +1 keeps DN==0 (nodata / outside swath) finite\n    at exactly 0.0 dB instead of -inf. This is NOT calibrated sigma0 - Cerulean\n    deliberately skips calibration [SRC].\n    \"\"\"\n    import numpy as np\n\n    dn = np.asarray(dn, dtype=np.float32)\n    return 10.0 * np.log10(dn * dn + 1.0)\n\n\n#: dB clip range used to scale to uint8. Computed once in Phase 2 from the\n#: 1st/99th percentiles of training tiles and then FROZEN. Until that\n#: measurement exists this is None and callers must fail loudly rather than\n#: guess - see DECISIONS.md.\nDB_CLIP: tuple[float, float] | None = None\n\n#: Dataset normalisation (mean/std of uint8/255 tiles). Measured in Phase 2 and\n#: written to model_card.json. None until measured.\nNORM_MEAN: float | None = None\nNORM_STD: float | None = None\n\n\n# ---------------------------------------------------------------------------\n# Classes\n# ---------------------------------------------------------------------------\n\n#: Our model's 4 output classes  [SRC]\nMODEL_CLASSES = {0: \"BACKGROUND\", 1: \"INFRA\", 2: \"NATURAL\", 3: \"VESSEL\"}\nN_CLASSES = 4\nOIL_CLASSES = (1, 2, 3)\nIGNORE_INDEX = 255\n\n#: Cerulean's public.cls ids  [DOC]\nCERULEAN_CLS = {\n    1: \"NOT_OIL\", 2: \"ANTHRO\", 3: \"NATURAL\", 4: \"INFRA\", 5: \"VESSEL\",\n    6: \"OLD_VESSEL\", 7: \"REC_VESSEL\", 8: \"COIN_VESSEL\", 9: \"AMBIGUOUS\",\n    10: \"LAND\", 11: \"SEA_ICE\", 12: \"ARTEFACT\",\n}\n\n#: Cerulean cls id -> our model class. 255 == ignore (excluded from loss).\nCLS_TO_MODEL: dict[int, int] = {\n    4: 1,                       # INFRA\n    3: 2,                       # NATURAL\n    5: 3, 6: 3, 7: 3, 8: 3,     # VESSEL family\n    2: IGNORE_INDEX,            # ANTHRO - too vague to supervise\n    9: IGNORE_INDEX,            # AMBIGUOUS\n    1: 0, 10: 0, 11: 0, 12: 0,  # NOT_OIL / LAND / SEA_ICE / ARTEFACT\n}\n#: Cerulean cls ids that become background but are informative negatives.\nHARD_NEGATIVE_CLS = (1, 10, 11, 12)\n#: Cerulean cls ids that map to a positive model class.\nPOSITIVE_CLS = tuple(sorted(k for k, v in CLS_TO_MODEL.items() if v in OIL_CLASSES))\n\n\n# ---------------------------------------------------------------------------\n# Post-processing thresholds  [SRC unless noted]\n# ---------------------------------------------------------------------------\n\n@dataclass(frozen=True)\nclass PostprocessConfig:\n    #: Hysteresis low threshold - islands of P(oil) above this are candidates.\n    p_low: float = 1e-4\n    #: Hysteresis seed threshold - an island survives only if some pixel >= this.\n    p_seed: float = 0.9\n    #: Polygon trim threshold.\n    p_trim: float = 0.5\n    #: Drop polygons >edge_frac within this many degrees of the scene edge.\n    edge_deg: float = 0.005\n    edge_frac: float = 0.5\n    #: Polygon NMS IoU.\n    nms_iou: float = 0.2\n    #: Minimum polygon area, m2. [OURS] - Cerulean's value is not published.\n    min_area_m2: float = 100_000.0\n    #: Simplify tolerance, in pixels.\n    simplify_px: float = 0.5\n    #: Land mask buffer, metres.\n    land_buffer_m: float = 1_000.0\n\n\nPOSTPROCESS = PostprocessConfig()\n\n#: Instance-matching IoU for evaluation. [OURS] - documented in reports.\nEVAL_INSTANCE_IOU = 0.1\n\n\n# ---------------------------------------------------------------------------\n# Dataset construction\n# ---------------------------------------------------------------------------\n\n#: Decimation factor for reads of the native ~10 m GRD onto our ~76 m grid.\n#: 8 takes 10 m -> 80 m, slightly coarser than the target, so `average`\n#: resampling in the warp does the final small step.\nREAD_DECIMATION = 8\n\n#: Tier-1 (human-reviewed) label weight in the training loss.\nT1_LOSS_WEIGHT = 3.0\n#: Minimum machine_confidence for a Tier-2 (unreviewed) label.\nT2_MIN_CONFIDENCE = 0.9\n#: Discard tiles with more than this fraction of nodata.\nMAX_NODATA_FRAC = 0.5\n#: Empty-ocean tiles are sampled at this ratio to positive tiles.\nNEGATIVE_RATIO = 1.0\n#: Empty tiles must be at least this far from any labelled polygon, metres.\nNEGATIVE_EXCLUSION_M = 5_000.0\n\n#: Split grouping cell size, degrees.\nSPLIT_CELL_DEG = 2.0\nTEST_CELL_FRAC = 0.15\nVAL_CELL_FRAC = 0.15\nSPLIT_SEED = 20260927\n\n#: Trailing months reserved as the temporal test set.\nTEMPORAL_TEST_MONTHS = 3\nDATASET_START_DATE = \"2023-01-01\"\n\n\n@dataclass(frozen=True)\nclass AOI:\n    name: str\n    #: (min_lon, min_lat, max_lon, max_lat)\n    bbox: tuple[float, float, float, float]\n\n\n#: Diverse ocean AOIs for dataset construction. [OURS]\nAOIS: tuple[AOI, ...] = (\n    AOI(\"arabian_sea_west_india\", (68.0, 6.0, 78.0, 20.0)),\n    AOI(\"persian_gulf\", (47.5, 23.5, 57.0, 30.5)),\n    AOI(\"gulf_of_mexico\", (-97.5, 18.0, -81.0, 30.5)),\n    AOI(\"east_mediterranean\", (25.0, 30.5, 36.5, 37.5)),\n    AOI(\"gulf_of_guinea\", (-2.0, -2.0, 9.0, 7.0)),\n    AOI(\"malacca_south_china_sea\", (99.0, -1.0, 115.0, 12.0)),\n    AOI(\"north_sea\", (-2.0, 51.0, 9.0, 60.0)),\n)\n\n#: Default AOI for the Phase 12 batch job.\nBATCH_AOI = AOIS[0]\n\n\n# ---------------------------------------------------------------------------\n# External services\n# ---------------------------------------------------------------------------\n\nCERULEAN_BASE = \"https://api.cerulean.skytruth.org\"\nCERULEAN_SLICKS = f\"{CERULEAN_BASE}/collections/public.slick_plus/items\"\nCERULEAN_SCENES = f\"{CERULEAN_BASE}/collections/public.sentinel1_grd/items\"\n#: Politeness budget for the Cerulean API.\nCERULEAN_MAX_RPS = 2.0\nCERULEAN_PAGE_LIMIT = 500\n\nPC_STAC = \"https://planetarycomputer.microsoft.com/api/stac/v1\"\nPC_COLLECTION = \"sentinel-1-grd\"\nPC_ASSET = \"vv\"\n\nNATURAL_EARTH_LAND_URL = (\n    \"https://naciscdn.org/naturalearth/10m/physical/ne_10m_land.zip\"\n)\n\n\n# ---------------------------------------------------------------------------\n# Database / API\n# ---------------------------------------------------------------------------\n\nDATABASE_URL = os.getenv(\"DATABASE_URL\", \"sqlite:///./oilspill.db\")\nFRONTEND_ORIGIN = os.getenv(\"FRONTEND_ORIGIN\", \"http://localhost:5173\")\nMAX_UPLOAD_BYTES = 200 * 1024 * 1024\n\n\n# ---------------------------------------------------------------------------\n# Attribution - must appear in README and the UI footer.\n# ---------------------------------------------------------------------------\n\nATTRIBUTION_CERULEAN = \"SkyTruth Cerulean (cerulean.skytruth.org). 2026. CC BY-SA 4.0.\"\nATTRIBUTION_COPERNICUS = \"Contains modified Copernicus Sentinel data 2026.\"\n", "src/oilspill/crsutil.py": "\"\"\"CRS transforms and metric geometry, routed through rasterio's bundled PROJ.\n\nWhy not pyproj or geopandas.to_crs? On this machine Windows Smart App Control\nblocks pyproj's and pyogrio's unsigned native extensions\n(\"An Application Control policy has blocked this file\"), so ``import pyproj``\nfails outright and ``GeoDataFrame.to_crs`` raises. rasterio's wheel bundles its\nown GDAL 3.12 / PROJ 9.8 and loads fine, so every coordinate transform in this\npackage goes through :func:`rasterio.warp.transform`. See DECISIONS.md D0.6.\n\nThis keeps the package working with *no* pyproj dependency at all, which also\nmakes the Docker image smaller.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport math\n\nfrom rasterio.crs import CRS\nfrom rasterio.warp import transform as _rio_transform\nfrom rasterio.warp import transform_geom as _rio_transform_geom\nfrom shapely.geometry import mapping, shape\nfrom shapely.geometry.base import BaseGeometry\n\nfrom . import config\n\nWGS84 = CRS.from_epsg(config.CRS_EPSG)\n\n\ndef estimate_utm_epsg(lon: float, lat: float) -> int:\n    \"\"\"EPSG code of the UTM zone containing a lon/lat point.\n\n    Replaces ``GeoSeries.estimate_utm_crs``. Northern zones are 326xx, southern\n    327xx. Longitude is normalised into [-180, 180) first so an antimeridian\n    centroid cannot produce zone 61.\n    \"\"\"\n    lon = ((lon + 180.0) % 360.0) - 180.0\n    zone = int((lon + 180.0) // 6.0) + 1\n    zone = min(max(zone, 1), 60)\n    return (32600 if lat >= 0 else 32700) + zone\n\n\ndef utm_crs_for(geom: BaseGeometry) -> CRS:\n    \"\"\"A local UTM CRS appropriate for measuring ``geom``.\"\"\"\n    c = geom.centroid\n    return CRS.from_epsg(estimate_utm_epsg(c.x, c.y))\n\n\ndef to_crs(geom: BaseGeometry, dst: CRS, src: CRS = WGS84) -> BaseGeometry:\n    \"\"\"Reproject a shapely geometry between two CRSs.\"\"\"\n    return shape(_rio_transform_geom(src, dst, mapping(geom)))\n\n\ndef transform_points(\n    xs, ys, dst: CRS, src: CRS = WGS84\n) -> tuple[list[float], list[float]]:\n    \"\"\"Reproject parallel coordinate sequences.\"\"\"\n    out_x, out_y = _rio_transform(src, dst, list(xs), list(ys))\n    return list(out_x), list(out_y)\n\n\ndef metric_properties(geom: BaseGeometry) -> dict[str, float]:\n    \"\"\"Area (m2), perimeter (m), length (m) and Polsby-Popper for a lon/lat geometry.\n\n    Measured in the geometry's local UTM zone, which is accurate to well under a\n    percent for slick-sized features and is what Cerulean's own ``area`` /\n    ``perimeter`` / ``polsby_popper`` fields are comparable against.\n\n    ``length`` is the major-axis extent, taken as the longer side of the minimum\n    rotated rectangle - the standard proxy for slick length, since a slick is a\n    long thin feature and its skeleton length is expensive to compute.\n    \"\"\"\n    utm = utm_crs_for(geom)\n    g = to_crs(geom, utm)\n\n    area = float(g.area)\n    perimeter = float(g.length)\n\n    try:\n        mrr = g.minimum_rotated_rectangle\n        coords = list(mrr.exterior.coords)[:5]\n        sides = [\n            math.dist(coords[i], coords[i + 1]) for i in range(min(4, len(coords) - 1))\n        ]\n        length = max(sides) if sides else 0.0\n        width = min(sides) if sides else 0.0\n    except Exception:  # noqa: BLE001 - degenerate geometries\n        length = width = 0.0\n\n    polsby_popper = (\n        4.0 * math.pi * area / (perimeter * perimeter) if perimeter > 0 else 0.0\n    )\n\n    return {\n        \"area_m2\": area,\n        \"perimeter_m\": perimeter,\n        \"length_m\": float(length),\n        \"width_m\": float(width),\n        \"polsby_popper\": float(polsby_popper),\n        \"utm_epsg\": int(utm.to_epsg()),\n    }\n\n\ndef buffer_metres(geom: BaseGeometry, metres: float) -> BaseGeometry:\n    \"\"\"Buffer a lon/lat geometry by a true ground distance.\n\n    Done by going out to local UTM, buffering there, and coming back - a degree\n    buffer would be anisotropic and wrong away from the equator.\n    \"\"\"\n    utm = utm_crs_for(geom)\n    return to_crs(to_crs(geom, utm).buffer(metres), WGS84, src=utm)\n\n\ndef metres_to_degrees(metres: float, lat: float) -> tuple[float, float]:\n    \"\"\"(dlon, dlat) spanning ``metres`` at latitude ``lat``.\n\n    Cheap approximation for padding windows, where isotropy matters more than\n    exactness. Use :func:`buffer_metres` when the result feeds a measurement.\n    \"\"\"\n    dlat = metres / 111_320.0\n    cos_lat = max(math.cos(math.radians(lat)), 0.01)\n    dlon = metres / (111_320.0 * cos_lat)\n    return dlon, dlat\n", "src/oilspill/georef.py": "\"\"\"Georeferencing: the Cerulean grid, and GCP-based warping of Sentinel-1 GRD.\n\nPlanetary Computer's ``sentinel-1-grd`` COGs carry no affine geotransform - only\na set of ground control points (~210 tie points). The STAC ``proj:transform`` is\na bbox approximation and must never be used for pixel georeferencing. Everything\nhere goes through ``rasterio.warp.reproject`` with explicit GCPs.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nfrom dataclasses import dataclass\n\nimport numpy as np\nimport rasterio\nfrom affine import Affine\nfrom rasterio.control import GroundControlPoint\nfrom rasterio.enums import Resampling\nfrom rasterio.warp import reproject\n\nfrom . import config\n\nlog = logging.getLogger(__name__)\n\n\n# ---------------------------------------------------------------------------\n# The grid\n# ---------------------------------------------------------------------------\n\n@dataclass(frozen=True)\nclass GridWindow:\n    \"\"\"A rectangular, grid-aligned window in EPSG:4326.\n\n    ``col_off``/``row_off`` are integer offsets from the global grid origin at\n    (GRID_ORIGIN_LON, GRID_ORIGIN_LAT), so two windows built independently are\n    always pixel-consistent with each other.\n    \"\"\"\n\n    col_off: int\n    row_off: int\n    width: int\n    height: int\n\n    @property\n    def transform(self) -> Affine:\n        px = config.PIXEL_DEG\n        return Affine(\n            px, 0.0, config.GRID_ORIGIN_LON + self.col_off * px,\n            0.0, -px, config.GRID_ORIGIN_LAT - self.row_off * px,\n        )\n\n    @property\n    def shape(self) -> tuple[int, int]:\n        return (self.height, self.width)\n\n    @property\n    def bounds(self) -> tuple[float, float, float, float]:\n        \"\"\"(min_lon, min_lat, max_lon, max_lat) of the window's outer edges.\"\"\"\n        px = config.PIXEL_DEG\n        west = config.GRID_ORIGIN_LON + self.col_off * px\n        north = config.GRID_ORIGIN_LAT - self.row_off * px\n        return (west, north - self.height * px, west + self.width * px, north)\n\n\ndef lon_to_col(lon: float) -> float:\n    \"\"\"Fractional global grid column of a longitude.\"\"\"\n    return (lon - config.GRID_ORIGIN_LON) / config.PIXEL_DEG\n\n\ndef lat_to_row(lat: float) -> float:\n    \"\"\"Fractional global grid row of a latitude.\"\"\"\n    return (config.GRID_ORIGIN_LAT - lat) / config.PIXEL_DEG\n\n\ndef grid_window_from_bounds(\n    bounds: tuple[float, float, float, float],\n    pad_m: float = 0.0,\n) -> GridWindow:\n    \"\"\"Snap a lon/lat bbox outward onto the grid, optionally padded.\n\n    ``pad_m`` is converted to degrees using the latitude of the bbox centre for\n    longitude, so the padding is roughly isotropic on the ground.\n    \"\"\"\n    min_lon, min_lat, max_lon, max_lat = bounds\n    if pad_m:\n        lat_mid = 0.5 * (min_lat + max_lat)\n        pad_lat = pad_m / 111_320.0\n        # Guard against the cosine collapsing near the poles.\n        cos_lat = max(np.cos(np.radians(lat_mid)), 0.01)\n        pad_lon = pad_m / (111_320.0 * cos_lat)\n        min_lon -= pad_lon\n        max_lon += pad_lon\n        min_lat -= pad_lat\n        max_lat += pad_lat\n\n    min_lat = max(min_lat, -90.0)\n    max_lat = min(max_lat, 90.0)\n\n    col0 = int(np.floor(lon_to_col(min_lon)))\n    col1 = int(np.ceil(lon_to_col(max_lon)))\n    # Rows increase southward, so max_lat gives the smaller row index.\n    row0 = int(np.floor(lat_to_row(max_lat)))\n    row1 = int(np.ceil(lat_to_row(min_lat)))\n    return GridWindow(col0, row0, max(col1 - col0, 1), max(row1 - row0, 1))\n\n\ndef assert_on_grid(transform: Affine, crs, tol: float = 1e-9) -> None:\n    \"\"\"Fail loudly unless a raster sits exactly on the Cerulean grid.\n\n    Called at every stage boundary so a CRS or half-pixel mistake cannot\n    silently propagate into the dataset or the detections.\n    \"\"\"\n    if crs is not None:\n        epsg = rasterio.crs.CRS.from_user_input(crs).to_epsg()\n        if epsg != config.CRS_EPSG:\n            raise AssertionError(f\"expected EPSG:{config.CRS_EPSG}, got EPSG:{epsg}\")\n\n    px = config.PIXEL_DEG\n    if abs(transform.a - px) > tol or abs(transform.e + px) > tol:\n        raise AssertionError(\n            f\"pixel size {transform.a}, {transform.e} != +/-{px}\"\n        )\n    if abs(transform.b) > tol or abs(transform.d) > tol:\n        raise AssertionError(f\"transform is rotated/sheared: {transform}\")\n\n    col = (transform.c - config.GRID_ORIGIN_LON) / px\n    row = (config.GRID_ORIGIN_LAT - transform.f) / px\n    if abs(col - round(col)) > 1e-6 or abs(row - round(row)) > 1e-6:\n        raise AssertionError(\n            f\"origin ({transform.c}, {transform.f}) is not grid-aligned \"\n            f\"(fractional col={col}, row={row})\"\n        )\n\n\n# ---------------------------------------------------------------------------\n# GCP handling\n# ---------------------------------------------------------------------------\n\ndef scale_gcps(\n    gcps: list[GroundControlPoint], factor: float\n) -> list[GroundControlPoint]:\n    \"\"\"Rescale GCP pixel coordinates for a decimated read.\n\n    A GCP's ``(col, row)`` refers to a *pixel corner* in the full-resolution\n    grid (GDAL convention). Under decimation by ``factor`` the corner at\n    full-res coordinate ``c`` lands at ``c / factor``, so the mapping is a plain\n    division - no +/-0.5 pixel-centre correction. See DECISIONS.md; the\n    alternative was tested in Phase 1 and produced a visible half-pixel shift.\n    \"\"\"\n    if factor == 1:\n        return list(gcps)\n    return [\n        GroundControlPoint(\n            row=g.row / factor, col=g.col / factor,\n            x=g.x, y=g.y, z=g.z, id=g.id, info=g.info,\n        )\n        for g in gcps\n    ]\n\n\n# ---------------------------------------------------------------------------\n# Warping\n# ---------------------------------------------------------------------------\n\ndef read_decimated(\n    ds: rasterio.DatasetReader,\n    factor: int = config.READ_DECIMATION,\n    band: int = 1,\n) -> tuple[np.ndarray, list[GroundControlPoint], rasterio.crs.CRS]:\n    \"\"\"Decimated whole-band read plus GCPs rescaled to match.\n\n    Uses ``out_shape`` so GDAL can serve the request from the COG's internal\n    overviews instead of pulling every full-resolution block.\n    \"\"\"\n    out_h = max(ds.height // factor, 1)\n    out_w = max(ds.width // factor, 1)\n    arr = ds.read(\n        band,\n        out_shape=(out_h, out_w),\n        resampling=Resampling.average,\n    )\n    gcps, gcp_crs = ds.gcps\n    if not gcps:\n        raise ValueError(\"dataset has no GCPs; cannot georeference\")\n    # Effective factor differs from `factor` when height/width are not exact\n    # multiples, and the two axes can differ. Scale each axis by its own ratio.\n    row_factor = ds.height / out_h\n    col_factor = ds.width / out_w\n    scaled = [\n        GroundControlPoint(\n            row=g.row / row_factor, col=g.col / col_factor,\n            x=g.x, y=g.y, z=g.z, id=g.id, info=g.info,\n        )\n        for g in gcps\n    ]\n    return arr, scaled, gcp_crs\n\n\ndef warp_to_grid(\n    src: np.ndarray,\n    gcps: list[GroundControlPoint],\n    gcp_crs,\n    window: GridWindow,\n    resampling: Resampling = Resampling.average,\n    src_nodata: float = config.NODATA,\n    dst_nodata: float = config.NODATA,\n    dtype: str = \"float32\",\n) -> np.ndarray:\n    \"\"\"Reproject a GCP-referenced array onto a Cerulean-grid window.\"\"\"\n    dst = np.full(window.shape, dst_nodata, dtype=dtype)\n    reproject(\n        source=src,\n        destination=dst,\n        gcps=gcps,\n        src_crs=gcp_crs,\n        src_nodata=src_nodata,\n        dst_transform=window.transform,\n        dst_crs=config.CRS,\n        dst_nodata=dst_nodata,\n        resampling=resampling,\n    )\n    assert_on_grid(window.transform, config.CRS)\n    return dst\n\n\ndef db_to_uint8(db: np.ndarray, clip: tuple[float, float]) -> np.ndarray:\n    \"\"\"Scale pseudo-dB to uint8 0-255 with a frozen clip range.\n\n    0 is reserved for nodata, matching Cerulean's convention that a zero input\n    pixel means \"no data\" [SRC]. Valid data is therefore mapped into 1-255.\n    \"\"\"\n    lo, hi = clip\n    if not hi > lo:\n        raise ValueError(f\"invalid dB clip range {clip}\")\n    valid = db > 0.0  # dn_to_db maps DN==0 to exactly 0.0\n    scaled = (db - lo) / (hi - lo)\n    out = np.clip(np.round(scaled * 254.0) + 1.0, 1, 255).astype(np.uint8)\n    out[~valid] = config.NODATA\n    return out\n\n\ndef write_grid_geotiff(path, arr: np.ndarray, window: GridWindow, dtype=None, nodata=None):\n    \"\"\"Write a single-band GeoTIFF carrying the exact grid transform.\"\"\"\n    from pathlib import Path\n\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    assert_on_grid(window.transform, config.CRS)\n    profile = {\n        \"driver\": \"GTiff\",\n        \"height\": arr.shape[0],\n        \"width\": arr.shape[1],\n        \"count\": 1,\n        \"dtype\": dtype or arr.dtype.name,\n        \"crs\": config.CRS,\n        \"transform\": window.transform,\n        \"compress\": \"deflate\",\n        \"predictor\": 2,\n        \"tiled\": True,\n        \"blockxsize\": 256,\n        \"blockysize\": 256,\n    }\n    if nodata is not None:\n        profile[\"nodata\"] = nodata\n    with rasterio.open(path, \"w\", **profile) as dst:\n        dst.write(arr, 1)\n    return path\n", "src/oilspill/s1_source.py": "\"\"\"Sentinel-1 GRD imagery from Microsoft Planetary Computer.\n\nFacts this module is built on (verified 2026-09-27):\n\n* STAC root ``https://planetarycomputer.microsoft.com/api/stac/v1``, collection\n  ``sentinel-1-grd``, VV band in asset ``vv``.\n* Signing is anonymous via ``planetary_computer.sign``; SAS tokens last about\n  24 h, so a 403 means \"re-sign and retry\", not \"gone\".\n* Assets are COGs of **raw DN** georeferenced **only by GCPs**. The STAC\n  ``proj:transform`` is a bbox approximation - see :mod:`oilspill.georef`.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport os\nimport random\nimport time\nfrom dataclasses import dataclass\nfrom datetime import UTC\nfrom pathlib import Path\nfrom typing import Any\n\nimport planetary_computer\nimport pystac_client\nimport rasterio\nfrom rasterio.errors import RasterioIOError\n\nfrom . import config\n\nlog = logging.getLogger(__name__)\n\n#: GDAL settings that make remote COG reads sane: no sidecar probing, a\n#: generous block cache, and retries on transient HTTP failures.\nGDAL_ENV = {\n    \"GDAL_DISABLE_READDIR_ON_OPEN\": \"EMPTY_DIR\",\n    \"CPL_VSIL_CURL_ALLOWED_EXTENSIONS\": \".tif,.tiff\",\n    \"GDAL_HTTP_MAX_RETRY\": \"5\",\n    \"GDAL_HTTP_RETRY_DELAY\": \"2\",\n    \"VSI_CACHE\": \"TRUE\",\n    \"VSI_CACHE_SIZE\": str(64 * 1024 * 1024),\n    \"GDAL_CACHEMAX\": \"512\",\n}\n\n\ndef apply_gdal_env() -> None:\n    \"\"\"Set the GDAL tuning variables in this process, without clobbering overrides.\"\"\"\n    for k, v in GDAL_ENV.items():\n        os.environ.setdefault(k, v)\n\n\n# ---------------------------------------------------------------------------\n# Scene identifiers\n# ---------------------------------------------------------------------------\n\n@dataclass(frozen=True)\nclass SceneKey:\n    \"\"\"The parts of a SAFE product name that identify an acquisition.\n\n    ``S1A_IW_GRDH_1SDV_20200912T032519_20200912T032544_034320_03FD6B_6C6C``\n      platform=S1A mode=IW product=GRDH polarisation=1SDV\n      start=20200912T032519 stop=20200912T032544\n      absolute_orbit=034320 mission_datatake=03FD6B unique_id=6C6C\n    \"\"\"\n\n    platform: str\n    mode: str\n    product: str\n    polarisation: str\n    start: str\n    stop: str\n\n    @property\n    def prefix(self) -> str:\n        \"\"\"The part that is stable across id spellings.\"\"\"\n        return \"_\".join(\n            (self.platform, self.mode, self.product, self.polarisation,\n             self.start, self.stop)\n        )\n\n\ndef parse_scene_id(scene_id: str) -> SceneKey | None:\n    \"\"\"Parse a SAFE product name. Returns None if it does not look like one.\"\"\"\n    parts = scene_id.strip().rstrip(\"/\").removesuffix(\".SAFE\").split(\"_\")\n    # Sentinel-1 product names use a doubled separator for the mode field in\n    # some spellings (e.g. S1A_IW_GRDH vs S1A_IW__GRDH), producing an empty part.\n    parts = [p for p in parts if p]\n    if len(parts) < 6 or not parts[0].startswith(\"S1\"):\n        return None\n    try:\n        return SceneKey(\n            platform=parts[0], mode=parts[1], product=parts[2],\n            polarisation=parts[3], start=parts[4], stop=parts[5],\n        )\n    except IndexError:\n        return None\n\n\n# ---------------------------------------------------------------------------\n# STAC\n# ---------------------------------------------------------------------------\n\n_catalog: pystac_client.Client | None = None\n\n\ndef catalog() -> pystac_client.Client:\n    \"\"\"Open (and memoise) the Planetary Computer STAC catalog.\"\"\"\n    global _catalog\n    if _catalog is None:\n        _catalog = pystac_client.Client.open(config.PC_STAC)\n    return _catalog\n\n\ndef _stac_cache_path(scene_id: str) -> Path:\n    config.STAC_CACHE_DIR.mkdir(parents=True, exist_ok=True)\n    return config.STAC_CACHE_DIR / f\"{scene_id}.json\"\n\n\ndef find_item(\n    scene_id: str,\n    intersects: dict[str, Any] | None = None,\n    use_cache: bool = True,\n) -> dict[str, Any] | None:\n    \"\"\"Locate the PC ``sentinel-1-grd`` item for a Cerulean ``s1_scene_id``.\n\n    Strategy, in order:\n\n    1. Exact item id lookup.\n    2. Exact lookup of the id with its trailing product-unique-id token removed.\n       **This is the case that actually fires.** Measured on 2026-09-27, all\n       three Phase-1 scenes showed Cerulean carrying one more token than PC::\n\n           Cerulean  S1A_..._050360_061036_7DC5\n           PC        S1A_..._050360_061036\n\n       so stripping the final ``_XXXX`` turns a 100-item search into one GET.\n    3. A datetime +/-2 min search (optionally constrained by ``intersects``),\n       matched on the product-id prefix, as a backstop.\n\n    Returns the item as a dict, or None.\n    \"\"\"\n    cache = _stac_cache_path(scene_id)\n    if use_cache and cache.exists():\n        try:\n            return json.loads(cache.read_text(encoding=\"utf-8\"))\n        except json.JSONDecodeError:\n            cache.unlink(missing_ok=True)\n\n    cat = catalog()\n    item_dict: dict[str, Any] | None = None\n\n    try:\n        collection = cat.get_collection(config.PC_COLLECTION)\n    except Exception as exc:  # noqa: BLE001\n        log.warning(\"cannot open collection %s: %s\", config.PC_COLLECTION, exc)\n        collection = None\n\n    # 1/2. exact id, then id minus the trailing unique-id token\n    candidates_ids = [scene_id]\n    head, sep, tail = scene_id.rpartition(\"_\")\n    if sep and len(tail) == 4:\n        candidates_ids.append(head)\n\n    if collection is not None:\n        for cand_id in candidates_ids:\n            try:\n                item = collection.get_item(cand_id)\n            except Exception as exc:  # noqa: BLE001 - STAC raises various errors\n                log.debug(\"id lookup failed for %s: %s\", cand_id, exc)\n                continue\n            if item is not None:\n                item_dict = item.to_dict()\n                how = \"exact\" if cand_id == scene_id else \"id_minus_suffix\"\n                log.info(\"matched %s -> %s (%s)\", scene_id, item.id, how)\n                break\n\n    # 2. datetime window + prefix match\n    if item_dict is None:\n        key = parse_scene_id(scene_id)\n        if key is None:\n            log.warning(\"cannot parse scene id %s\", scene_id)\n            return None\n        start = _iso(key.start)\n        lo = _shift_iso(start, -120)\n        hi = _shift_iso(start, +120)\n        search_kwargs: dict[str, Any] = {\n            \"collections\": [config.PC_COLLECTION],\n            \"datetime\": f\"{lo}/{hi}\",\n            \"limit\": 100,\n        }\n        if intersects is not None:\n            search_kwargs[\"intersects\"] = intersects\n        try:\n            candidates = list(cat.search(**search_kwargs).items())\n        except Exception as exc:  # noqa: BLE001\n            log.warning(\"STAC search failed for %s: %s\", scene_id, exc)\n            candidates = []\n\n        log.info(\n            \"exact id missed for %s; %d candidates in +/-2 min window\",\n            scene_id, len(candidates),\n        )\n        for cand in candidates:\n            cand_key = parse_scene_id(cand.id)\n            if cand_key is not None and cand_key.prefix == key.prefix:\n                item_dict = cand.to_dict()\n                log.info(\"matched %s -> %s by product-id prefix\", scene_id, cand.id)\n                break\n        else:\n            for cand in candidates:\n                log.info(\"   candidate id: %s\", cand.id)\n\n    if item_dict is not None:\n        cache.write_text(json.dumps(item_dict), encoding=\"utf-8\")\n    return item_dict\n\n\ndef _iso(compact: str) -> str:\n    \"\"\"``20200912T032519`` -> ``2020-09-12T03:25:19Z``.\"\"\"\n    d, t = compact.split(\"T\")\n    return f\"{d[0:4]}-{d[4:6]}-{d[6:8]}T{t[0:2]}:{t[2:4]}:{t[4:6]}Z\"\n\n\ndef _shift_iso(iso: str, seconds: int) -> str:\n    from datetime import datetime, timedelta\n\n    dt = datetime.strptime(iso, \"%Y-%m-%dT%H:%M:%SZ\").replace(tzinfo=UTC)\n    return (dt + timedelta(seconds=seconds)).strftime(\"%Y-%m-%dT%H:%M:%SZ\")\n\n\ndef sign_asset(item: dict[str, Any], asset: str = config.PC_ASSET) -> str:\n    \"\"\"Return a freshly signed href for one asset of an item.\"\"\"\n    assets = item.get(\"assets\", {})\n    if asset not in assets:\n        raise KeyError(\n            f\"item {item.get('id')} has no asset {asset!r}; \"\n            f\"available: {sorted(assets)}\"\n        )\n    return planetary_computer.sign(assets[asset][\"href\"])\n\n\ndef open_asset(\n    item: dict[str, Any],\n    asset: str = config.PC_ASSET,\n    max_retries: int = 4,\n) -> rasterio.DatasetReader:\n    \"\"\"Open a signed asset, re-signing on 403 (expired SAS token).\n\n    The caller owns the returned dataset and must close it.\n    \"\"\"\n    apply_gdal_env()\n    last_exc: Exception | None = None\n    for attempt in range(max_retries):\n        href = sign_asset(item, asset)\n        try:\n            return rasterio.open(href)\n        except RasterioIOError as exc:\n            last_exc = exc\n            msg = str(exc)\n            transient = \"403\" in msg or \"404\" in msg or \"timed out\" in msg.lower()\n            log.warning(\n                \"open failed (attempt %d/%d, transient=%s): %s\",\n                attempt + 1, max_retries, transient, msg.splitlines()[0],\n            )\n            if not transient:\n                raise\n            time.sleep(min(2.0**attempt, 20.0) + random.uniform(0, 1))\n    raise RuntimeError(f\"could not open asset {asset}: {last_exc}\") from last_exc\n\n\ndef search_scenes(\n    bbox: tuple[float, float, float, float],\n    datetime_range: str,\n    mode: str = \"IW\",\n    max_items: int | None = None,\n) -> list[dict[str, Any]]:\n    \"\"\"List ``sentinel-1-grd`` items over a bbox and time range.\n\n    Filters to the requested acquisition mode and to items that actually carry a\n    ``vv`` asset (so SH/single-pol products are dropped).\n    \"\"\"\n    search = catalog().search(\n        collections=[config.PC_COLLECTION],\n        bbox=list(bbox),\n        datetime=datetime_range,\n        limit=500,\n    )\n    out: list[dict[str, Any]] = []\n    for item in search.items():\n        key = parse_scene_id(item.id)\n        if key is not None and key.mode != mode:\n            continue\n        d = item.to_dict()\n        if config.PC_ASSET not in d.get(\"assets\", {}):\n            continue\n        out.append(d)\n        if max_items is not None and len(out) >= max_items:\n            break\n    return out\n\n\ndef item_footprint(item: dict[str, Any]) -> dict[str, Any]:\n    \"\"\"The item's geometry, as a GeoJSON geometry dict.\"\"\"\n    geom = item.get(\"geometry\")\n    if geom is None:\n        raise ValueError(f\"item {item.get('id')} has no geometry\")\n    return geom\n", "src/oilspill/cerulean_api.py": "\"\"\"Client for the Cerulean OGC API Features endpoint (tipg).\n\nPoliteness: <= ``config.CERULEAN_MAX_RPS`` requests/second, exponential backoff\nwith jitter on 429/5xx, and every response cached to disk so a re-run costs\nnothing.\n\nVerified against the live service on 2026-09-27:\n\n* ``/conformance`` advertises ``ogcapi-features-3/1.0/conf/filter``, so CQL2 is\n  supported. ``filter-lang=cql2-text`` works.\n* **``hitl_cls IS NOT NULL`` is silently ignored** - it and ``IS NULL`` both\n  return the same ``numberMatched`` (1,326,855 of 1,334,040). Comparison\n  operators do work: ``hitl_cls > 0`` returns 7,185 and every row really does\n  carry a hitl value. So ``hitl_cls > 0`` is our Tier-1 predicate. See\n  DECISIONS.md.\n* ``s1_scene_id`` is the full SAFE product name, e.g.\n  ``S1A_IW_GRDH_1SDV_20200912T032519_20200912T032544_034320_03FD6B_6C6C``.\n* ``area`` is in square metres.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport logging\nimport random\nimport threading\nimport time\nfrom collections.abc import Iterator\nfrom pathlib import Path\nfrom typing import Any\nfrom urllib.parse import urlencode\n\nimport httpx\n\nfrom . import config\n\nlog = logging.getLogger(__name__)\n\n#: Cerulean cls ids whose slicks we treat as positive labels, as a CQL2 list.\n_POSITIVE_CLS_SQL = \", \".join(str(c) for c in config.POSITIVE_CLS)\n\n\nclass _RateLimiter:\n    \"\"\"Minimum-interval limiter, safe across threads.\"\"\"\n\n    def __init__(self, max_rps: float):\n        self._min_interval = 1.0 / max_rps if max_rps > 0 else 0.0\n        self._lock = threading.Lock()\n        self._next_allowed = 0.0\n\n    def wait(self) -> None:\n        if self._min_interval <= 0:\n            return\n        with self._lock:\n            now = time.monotonic()\n            sleep_for = self._next_allowed - now\n            self._next_allowed = max(now, self._next_allowed) + self._min_interval\n        if sleep_for > 0:\n            time.sleep(sleep_for)\n\n\n_limiter = _RateLimiter(config.CERULEAN_MAX_RPS)\n\n\nclass CeruleanClient:\n    \"\"\"Cached, rate-limited reader for the Cerulean API.\"\"\"\n\n    def __init__(\n        self,\n        cache_dir: Path | None = None,\n        timeout: float = 120.0,\n        max_retries: int = 5,\n        use_cache: bool = True,\n    ):\n        self.cache_dir = Path(cache_dir or config.CERULEAN_CACHE_DIR)\n        self.cache_dir.mkdir(parents=True, exist_ok=True)\n        self.use_cache = use_cache\n        self.max_retries = max_retries\n        self._client = httpx.Client(\n            timeout=timeout,\n            follow_redirects=True,\n            headers={\"User-Agent\": \"oilspill/0.1 (open-source Cerulean reproduction)\"},\n        )\n\n    # -- plumbing ----------------------------------------------------------\n\n    def _cache_path(self, url: str, params: dict[str, Any]) -> Path:\n        key = f\"{url}?{urlencode(sorted(params.items()))}\"\n        digest = hashlib.sha256(key.encode()).hexdigest()[:24]\n        return self.cache_dir / f\"{digest}.json\"\n\n    def get(self, url: str, params: dict[str, Any]) -> dict[str, Any]:\n        \"\"\"GET with disk cache, rate limiting and backoff.\"\"\"\n        cache_path = self._cache_path(url, params)\n        if self.use_cache and cache_path.exists():\n            try:\n                return json.loads(cache_path.read_text(encoding=\"utf-8\"))\n            except json.JSONDecodeError:\n                log.warning(\"corrupt cache entry %s; refetching\", cache_path.name)\n                cache_path.unlink(missing_ok=True)\n\n        last_exc: Exception | None = None\n        for attempt in range(self.max_retries):\n            _limiter.wait()\n            try:\n                resp = self._client.get(url, params=params)\n            except httpx.HTTPError as exc:\n                # The service resets connections on some queries; treat a\n                # transport failure exactly like a 5xx and back off.\n                last_exc = exc\n                log.warning(\"transport error (attempt %d): %s\", attempt + 1, exc)\n            else:\n                if resp.status_code == 200:\n                    payload = resp.json()\n                    cache_path.write_text(\n                        json.dumps(payload), encoding=\"utf-8\"\n                    )\n                    return payload\n                if resp.status_code in (429,) or resp.status_code >= 500:\n                    last_exc = httpx.HTTPStatusError(\n                        f\"HTTP {resp.status_code}\", request=resp.request, response=resp\n                    )\n                    log.warning(\n                        \"HTTP %d from Cerulean (attempt %d)\",\n                        resp.status_code, attempt + 1,\n                    )\n                else:\n                    resp.raise_for_status()\n\n            backoff = min(2.0**attempt, 60.0) + random.uniform(0.0, 1.0)\n            log.info(\"backing off %.1fs\", backoff)\n            time.sleep(backoff)\n\n        raise RuntimeError(\n            f\"Cerulean API failed after {self.max_retries} attempts: {last_exc}\"\n        ) from last_exc\n\n    # -- queries -----------------------------------------------------------\n\n    def iter_slicks(\n        self,\n        cql2: str | None = None,\n        bbox: tuple[float, float, float, float] | None = None,\n        datetime_range: str | None = None,\n        sortby: str | None = None,\n        limit: int = config.CERULEAN_PAGE_LIMIT,\n        max_features: int | None = None,\n    ) -> Iterator[dict[str, Any]]:\n        \"\"\"Yield slick GeoJSON features, paging with limit/offset.\n\n        ``cql2`` is CQL2-text. Remember that ``IS NULL`` / ``IS NOT NULL`` do not\n        work on this deployment - use ``hitl_cls > 0`` for reviewed slicks.\n        \"\"\"\n        offset = 0\n        yielded = 0\n        while True:\n            page_limit = limit\n            if max_features is not None:\n                page_limit = min(limit, max_features - yielded)\n                if page_limit <= 0:\n                    return\n\n            params: dict[str, Any] = {\"limit\": page_limit, \"offset\": offset}\n            if cql2:\n                params[\"filter-lang\"] = \"cql2-text\"\n                params[\"filter\"] = cql2\n            if bbox:\n                params[\"bbox\"] = \",\".join(str(v) for v in bbox)\n            if datetime_range:\n                params[\"datetime\"] = datetime_range\n            if sortby:\n                params[\"sortby\"] = sortby\n\n            payload = self.get(config.CERULEAN_SLICKS, params)\n            features = payload.get(\"features\", [])\n            if not features:\n                return\n            for feat in features:\n                yield feat\n                yielded += 1\n                if max_features is not None and yielded >= max_features:\n                    return\n            # The server may cap the page size below what we asked for (tipg does),\n            # so follow its \"next\" link rather than comparing against page_limit.\n            has_next = any(link.get(\"rel\") == \"next\" for link in payload.get(\"links\", []))\n            if not has_next:\n                return\n            offset += len(features)\n\n    def slicks_in_scene(self, scene_id: str) -> list[dict[str, Any]]:\n        \"\"\"Every slick of any class in one scene.\n\n        Used so that an unlabelled Cerulean slick is never silently treated as\n        background when building masks.\n        \"\"\"\n        escaped = scene_id.replace(\"'\", \"''\")\n        return list(self.iter_slicks(cql2=f\"s1_scene_id = '{escaped}'\"))\n\n    def reviewed_slicks(\n        self,\n        min_area_m2: float = 5.0e6,\n        min_confidence: float = 0.9,\n        bbox: tuple[float, float, float, float] | None = None,\n        datetime_range: str | None = None,\n        max_features: int | None = None,\n        sortby: str | None = None,\n    ) -> list[dict[str, Any]]:\n        \"\"\"Tier-1 slicks: human-reviewed (``hitl_cls > 0``) and a positive class.\"\"\"\n        cql2 = (\n            f\"hitl_cls IN ({_POSITIVE_CLS_SQL}) \"\n            f\"AND area > {min_area_m2} \"\n            f\"AND machine_confidence > {min_confidence}\"\n        )\n        return list(\n            self.iter_slicks(\n                cql2=cql2, bbox=bbox, datetime_range=datetime_range,\n                max_features=max_features, sortby=sortby,\n            )\n        )\n\n    def unreviewed_slicks(\n        self,\n        min_confidence: float = config.T2_MIN_CONFIDENCE,\n        bbox: tuple[float, float, float, float] | None = None,\n        datetime_range: str | None = None,\n        max_features: int | None = None,\n    ) -> list[dict[str, Any]]:\n        \"\"\"Tier-2 slicks: high machine confidence, positive class.\n\n        ``hitl_cls`` cannot be tested for NULL server-side, so the reviewed rows\n        are filtered out client-side.\n        \"\"\"\n        cql2 = (\n            f\"cls IN ({_POSITIVE_CLS_SQL}) \"\n            f\"AND machine_confidence >= {min_confidence}\"\n        )\n        feats = self.iter_slicks(\n            cql2=cql2, bbox=bbox, datetime_range=datetime_range,\n            max_features=max_features,\n        )\n        return [f for f in feats if not f[\"properties\"].get(\"hitl_cls\")]\n\n    def close(self) -> None:\n        self._client.close()\n\n    def __enter__(self) -> CeruleanClient:\n        return self\n\n    def __exit__(self, *exc) -> None:\n        self.close()\n\n\ndef effective_cls(props: dict[str, Any]) -> int:\n    \"\"\"Cerulean class actually used for labelling: ``hitl_cls`` if reviewed.\"\"\"\n    hitl = props.get(\"hitl_cls\")\n    return int(hitl) if hitl else int(props[\"cls\"])\n\n\ndef label_tier(props: dict[str, Any]) -> str | None:\n    \"\"\"``\"T1\"`` (reviewed), ``\"T2\"`` (confident machine label), or None to discard.\"\"\"\n    if props.get(\"hitl_cls\"):\n        return \"T1\"\n    if (props.get(\"machine_confidence\") or 0.0) >= config.T2_MIN_CONFIDENCE:\n        return \"T2\"\n    return None\n\n\ndef model_class(props: dict[str, Any]) -> int:\n    \"\"\"Our 0-3 model class (or ``config.IGNORE_INDEX``) for a slick.\"\"\"\n    return config.CLS_TO_MODEL.get(effective_cls(props), config.IGNORE_INDEX)\n", "src/oilspill/labels.py": "\"\"\"Turning Cerulean slick polygons into rasterised label masks on our grid.\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nfrom collections.abc import Iterable\nfrom typing import Any\n\nimport numpy as np\nfrom rasterio.features import rasterize\nfrom shapely.geometry import shape\nfrom shapely.geometry.base import BaseGeometry\nfrom shapely.ops import unary_union\n\nfrom . import config\nfrom .cerulean_api import label_tier, model_class\nfrom .georef import GridWindow\n\nlog = logging.getLogger(__name__)\n\n\ndef clean_geometry(geom: BaseGeometry) -> BaseGeometry | None:\n    \"\"\"Repair a geometry, or return None if nothing usable survives.\"\"\"\n    if geom is None or geom.is_empty:\n        return None\n    if not geom.is_valid:\n        from shapely import make_valid\n\n        geom = make_valid(geom)\n        if geom.is_empty:\n            return None\n    if geom.geom_type not in (\"Polygon\", \"MultiPolygon\", \"GeometryCollection\"):\n        return None\n    if geom.geom_type == \"GeometryCollection\":\n        polys = [g for g in geom.geoms if g.geom_type in (\"Polygon\", \"MultiPolygon\")]\n        if not polys:\n            return None\n        geom = unary_union(polys)\n    if geom.is_empty or geom.area <= 0:\n        return None\n    return geom\n\n\ndef slick_geometry(feature: dict[str, Any]) -> BaseGeometry | None:\n    \"\"\"The cleaned shapely geometry of one Cerulean slick feature.\"\"\"\n    if feature.get(\"geometry\") is None:\n        return None\n    return clean_geometry(shape(feature[\"geometry\"]))\n\n\ndef rasterize_slicks(\n    features: Iterable[dict[str, Any]],\n    window: GridWindow,\n) -> tuple[np.ndarray, dict[str, int]]:\n    \"\"\"Burn slick features into a label mask on ``window``.\n\n    Returns ``(mask, stats)``. Mask values are ``{0,1,2,3,255}`` where 255 is\n    ignore. Painting order matters, so it is fixed and explicit:\n\n    1. background (the initial fill),\n    2. positive classes 1-3,\n    3. ignore (255) last, so an AMBIGUOUS/ANTHRO slick overlapping a confident\n       one cannot silently downgrade it to ignore... and conversely a positive\n       label never overwrites an ignore region it fully contains.\n\n    Hard negatives (NOT_OIL / LAND / SEA_ICE / ARTEFACT) stay 0 but are counted.\n    \"\"\"\n    mask = np.zeros(window.shape, dtype=np.uint8)\n    stats = {\"positive\": 0, \"ignore\": 0, \"hard_negative\": 0, \"dropped\": 0, \"T1\": 0, \"T2\": 0}\n\n    positive: list[tuple[BaseGeometry, int]] = []\n    ignore: list[tuple[BaseGeometry, int]] = []\n\n    for feat in features:\n        props = feat.get(\"properties\", {})\n        geom = slick_geometry(feat)\n        if geom is None:\n            stats[\"dropped\"] += 1\n            continue\n\n        tier = label_tier(props)\n        cls = model_class(props)\n\n        if cls == config.IGNORE_INDEX:\n            ignore.append((geom, config.IGNORE_INDEX))\n            stats[\"ignore\"] += 1\n            continue\n        if cls == 0:\n            # Hard negative: correctly background, but worth counting/forcing\n            # into the tile sample.\n            stats[\"hard_negative\"] += 1\n            continue\n        if tier is None:\n            # Not reviewed and not confident enough - neither a label nor a\n            # trustworthy negative, so mark it ignore rather than background.\n            ignore.append((geom, config.IGNORE_INDEX))\n            stats[\"ignore\"] += 1\n            continue\n\n        positive.append((geom, cls))\n        stats[\"positive\"] += 1\n        stats[tier] += 1\n\n    if positive:\n        rasterize(\n            positive, out=mask, transform=window.transform,\n            all_touched=False, default_value=1,\n        )\n    if ignore:\n        rasterize(\n            ignore, out=mask, transform=window.transform,\n            all_touched=False, default_value=config.IGNORE_INDEX,\n        )\n    return mask, stats\n\n\ndef tier1_mask(\n    features: Iterable[dict[str, Any]],\n    window: GridWindow,\n) -> np.ndarray:\n    \"\"\"Mask built from Tier-1 (human-reviewed) slicks only, for evaluation.\n\n    Everything that is not a reviewed positive becomes ignore, so unreviewed\n    machine labels never count as either a hit or a false positive.\n    \"\"\"\n    mask = np.full(window.shape, config.IGNORE_INDEX, dtype=np.uint8)\n    feats = list(features)\n\n    reviewed_pos: list[tuple[BaseGeometry, int]] = []\n    for feat in feats:\n        props = feat.get(\"properties\", {})\n        if label_tier(props) != \"T1\":\n            continue\n        cls = model_class(props)\n        if cls in config.OIL_CLASSES:\n            geom = slick_geometry(feat)\n            if geom is not None:\n                reviewed_pos.append((geom, cls))\n\n    # Only scenes with at least one reviewed slick are usable for T1 metrics.\n    if not reviewed_pos:\n        return mask\n\n    mask[:] = 0\n    unreviewed: list[tuple[BaseGeometry, int]] = []\n    for feat in feats:\n        props = feat.get(\"properties\", {})\n        if label_tier(props) == \"T1\":\n            continue\n        geom = slick_geometry(feat)\n        if geom is not None:\n            unreviewed.append((geom, config.IGNORE_INDEX))\n\n    if unreviewed:\n        rasterize(\n            unreviewed, out=mask, transform=window.transform, all_touched=False,\n        )\n    rasterize(\n        reviewed_pos, out=mask, transform=window.transform, all_touched=False,\n    )\n    return mask\n", "src/oilspill/landmask.py": "\"\"\"Natural Earth 10 m land polygons, read without OGR.\n\n``geopandas.read_file`` needs pyogrio or fiona, and on this machine Smart App\nControl blocks pyogrio's native extension (DECISIONS.md D0.6). Natural Earth\nships plain shapefiles, so we read them with pyshp - pure Python, no native code\nand no GDAL/OGR involved at all.\n\nNatural Earth is public domain.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport io\nimport logging\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nfrom shapely.geometry import box, shape\nfrom shapely.geometry.base import BaseGeometry\nfrom shapely.strtree import STRtree\n\nfrom . import config\n\nlog = logging.getLogger(__name__)\n\n_LAND_ZIP = \"ne_10m_land.zip\"\n_cache: tuple[list[BaseGeometry], STRtree] | None = None\n\n\ndef land_zip_path() -> Path:\n    return config.CACHE_DIR / _LAND_ZIP\n\n\ndef download_land(force: bool = False) -> Path:\n    \"\"\"Fetch ne_10m_land.zip into the cache. Idempotent.\"\"\"\n    path = land_zip_path()\n    if path.exists() and not force and path.stat().st_size > 0:\n        return path\n    path.parent.mkdir(parents=True, exist_ok=True)\n    import httpx\n\n    log.info(\"downloading Natural Earth 10m land -> %s\", path)\n    with httpx.Client(timeout=300, follow_redirects=True) as c:\n        r = c.get(config.NATURAL_EARTH_LAND_URL)\n        r.raise_for_status()\n        path.write_bytes(r.content)\n    log.info(\"downloaded %.1f MB\", path.stat().st_size / 1e6)\n    return path\n\n\ndef _read_shapefile_from_zip(zip_path: Path) -> list[BaseGeometry]:\n    \"\"\"All polygon geometries from the single shapefile inside a Natural Earth zip.\"\"\"\n    import shapefile  # pyshp\n\n    with zipfile.ZipFile(zip_path) as zf:\n        names = zf.namelist()\n        shp = next((n for n in names if n.lower().endswith(\".shp\")), None)\n        if shp is None:\n            raise ValueError(f\"no .shp inside {zip_path}\")\n        stem = shp[: -len(\".shp\")]\n        dbf = next((n for n in names if n.lower() == f\"{stem.lower()}.dbf\"), None)\n        shx = next((n for n in names if n.lower() == f\"{stem.lower()}.shx\"), None)\n\n        kwargs = {\"shp\": io.BytesIO(zf.read(shp))}\n        if dbf:\n            kwargs[\"dbf\"] = io.BytesIO(zf.read(dbf))\n        if shx:\n            kwargs[\"shx\"] = io.BytesIO(zf.read(shx))\n\n        geoms: list[BaseGeometry] = []\n        with shapefile.Reader(**kwargs) as reader:\n            for sh in reader.iterShapes():\n                try:\n                    g = shape(sh.__geo_interface__)\n                except Exception as exc:  # noqa: BLE001\n                    log.debug(\"skipping unreadable shape: %s\", exc)\n                    continue\n                if g.is_empty:\n                    continue\n                if not g.is_valid:\n                    from shapely import make_valid\n\n                    g = make_valid(g)\n                    if g.is_empty:\n                        continue\n                geoms.append(g)\n    log.info(\"read %d land polygons from %s\", len(geoms), zip_path.name)\n    return geoms\n\n\ndef load_land() -> tuple[list[BaseGeometry], STRtree]:\n    \"\"\"Load (and memoise) land polygons plus a spatial index.\"\"\"\n    global _cache\n    if _cache is None:\n        geoms = _read_shapefile_from_zip(download_land())\n        _cache = (geoms, STRtree(geoms))\n    return _cache\n\n\ndef land_in_bounds(bounds: tuple[float, float, float, float]) -> list[BaseGeometry]:\n    \"\"\"Land polygons intersecting a lon/lat bbox.\"\"\"\n    geoms, tree = load_land()\n    query = box(*bounds)\n    return [geoms[i] for i in tree.query(query) if geoms[i].intersects(query)]\n\n\ndef land_mask(window, bounds_only: bool = False) -> np.ndarray:\n    \"\"\"Rasterise Natural Earth land onto a :class:`~oilspill.georef.GridWindow`.\"\"\"\n    from rasterio.features import rasterize\n\n    geoms = land_in_bounds(window.bounds)\n    mask = np.zeros(window.shape, dtype=bool)\n    if not geoms:\n        return mask\n    burned = rasterize(\n        [(g, 1) for g in geoms],\n        out_shape=window.shape,\n        transform=window.transform,\n        fill=0,\n        dtype=\"uint8\",\n        all_touched=True,\n    )\n    return burned.astype(bool)\n\n\ndef buffered_land(\n    bounds: tuple[float, float, float, float],\n    buffer_m: float = config.POSTPROCESS.land_buffer_m,\n) -> BaseGeometry | None:\n    \"\"\"Union of land in ``bounds``, buffered seaward by ``buffer_m``.\n\n    Used to drop detections that are really coastline artefacts. Cerulean applies\n    a 1 km land buffer for the same reason [SRC].\n    \"\"\"\n    from shapely.ops import unary_union\n\n    from .crsutil import buffer_metres\n\n    geoms = land_in_bounds(bounds)\n    if not geoms:\n        return None\n    union = unary_union(geoms)\n    if union.is_empty:\n        return None\n    try:\n        return buffer_metres(union, buffer_m)\n    except Exception as exc:  # noqa: BLE001\n        log.warning(\"metric land buffer failed (%s); falling back to degrees\", exc)\n        lat_mid = 0.5 * (bounds[1] + bounds[3])\n        from .crsutil import metres_to_degrees\n\n        dlon, _ = metres_to_degrees(buffer_m, lat_mid)\n        return union.buffer(dlon)\n", "src/oilspill/scene.py": "\"\"\"Whole-scene preparation: PC GRD item -> pseudo-dB raster on the Cerulean grid.\n\nThis is the ONE code path that turns imagery into model input. The dataset\nbuilder, the evaluation script, the API and the batch job all call\n:func:`prepare_item`, so training and inference can never drift apart.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nimport time\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport rasterio\nfrom rasterio.enums import Resampling\nfrom rasterio.warp import reproject, transform_bounds\n\nfrom . import config\nfrom .georef import GridWindow, grid_window_from_bounds, read_decimated, warp_to_grid\nfrom .s1_source import open_asset\n\nlog = logging.getLogger(__name__)\n\n#: Storage encoding of pseudo-dB rasters: uint16 = round(dB * DB_SCALE); 0 = nodata.\nDB_SCALE = 100.0\n\n\n@dataclass\nclass SceneRaster:\n    scene_id: str\n    db: np.ndarray            # float32 pseudo-dB, 0.0 == nodata\n    window: GridWindow\n    meta: dict[str, Any] = field(default_factory=dict)\n\n    @property\n    def valid(self) -> np.ndarray:\n        return self.db > 0.0\n\n    @property\n    def bounds(self) -> tuple[float, float, float, float]:\n        return self.window.bounds\n\n\ndef item_bounds(item: dict[str, Any]) -> tuple[float, float, float, float]:\n    bbox = item.get(\"bbox\")\n    if bbox and len(bbox) == 4:\n        return tuple(float(v) for v in bbox)  # type: ignore[return-value]\n    from shapely.geometry import shape\n\n    return tuple(shape(item[\"geometry\"]).bounds)  # type: ignore[return-value]\n\n\ndef prepare_item(item: dict[str, Any], decimation: int = config.READ_DECIMATION) -> SceneRaster:\n    \"\"\"Read a whole PC ``sentinel-1-grd`` VV asset and warp it onto the grid.\"\"\"\n    t0 = time.perf_counter()\n    ds = open_asset(item)\n    try:\n        arr, gcps, gcp_crs = read_decimated(ds, factor=decimation)\n        native = (ds.height, ds.width)\n    finally:\n        ds.close()\n    t_read = time.perf_counter() - t0\n\n    window = grid_window_from_bounds(item_bounds(item), pad_m=2_000.0)\n    t1 = time.perf_counter()\n    dn = warp_to_grid(arr.astype(\"float32\"), gcps, gcp_crs, window)\n    db = config.dn_to_db(dn)\n    db[dn <= 0] = 0.0\n    t_warp = time.perf_counter() - t1\n\n    props = item.get(\"properties\", {})\n    meta = {\n        \"native_shape\": native,\n        \"read_s\": round(t_read, 2),\n        \"warp_s\": round(t_warp, 2),\n        \"datetime\": props.get(\"datetime\") or props.get(\"start_datetime\"),\n        \"orbit_direction\": props.get(\"sat:orbit_state\"),\n        \"platform\": props.get(\"platform\"),\n    }\n    log.info(\"prepared %s shape=%s read=%.1fs warp=%.1fs\", item.get(\"id\"), db.shape, t_read, t_warp)\n    return SceneRaster(scene_id=item[\"id\"], db=db.astype(\"float32\"), window=window, meta=meta)\n\n\ndef prepare_geotiff(path: str | Path) -> SceneRaster:\n    \"\"\"Prepare a user-supplied Sentinel-1 VV GeoTIFF.\n\n    Accepted inputs (anything else raises ``ValueError``):\n\n    * raw GRD measurement TIFF / COG (integer DN) georeferenced by GCPs\n      (e.g. ``measurement/*-vv-*.tiff`` from a SAFE, or a PC ``vv`` asset);\n    * integer DN GeoTIFF with an affine transform in any CRS;\n    * float pseudo-dB GeoTIFF produced by this project (values ~20-60, 0 = nodata).\n\n    Calibrated sigma0 in dB (negative values, e.g. Earth Engine exports) is\n    rejected because the model was trained on uncalibrated pseudo-dB.\n    \"\"\"\n    path = Path(path)\n    with rasterio.open(path) as ds:\n        if ds.count < 1:\n            raise ValueError(\"GeoTIFF has no bands\")\n        gcps, gcp_crs = ds.gcps\n        has_transform = ds.crs is not None and not ds.transform.is_identity\n        if not gcps and not has_transform:\n            raise ValueError(\"GeoTIFF is not georeferenced (no CRS/transform and no GCPs)\")\n        is_int = np.issubdtype(np.dtype(ds.dtypes[0]), np.integer)\n\n        if gcps and not has_transform:\n            # Full-resolution GRD (~10 m) -> decimate like the PC path; small files as-is.\n            factor = config.READ_DECIMATION if max(ds.width, ds.height) > 8000 else 1\n            arr, sgcps, gcrs = read_decimated(ds, factor=factor)\n            from shapely.geometry import MultiPoint\n\n            b = MultiPoint([(g.x, g.y) for g in gcps]).bounds\n            window = grid_window_from_bounds(b, pad_m=1_000.0)\n            src = warp_to_grid(arr.astype(\"float32\"), sgcps, gcrs, window)\n        else:\n            b = transform_bounds(ds.crs, config.CRS, *ds.bounds)\n            window = grid_window_from_bounds(b)\n            nodata = ds.nodata if ds.nodata is not None else 0\n            src = np.zeros(window.shape, dtype=\"float32\")\n            reproject(\n                source=rasterio.band(ds, 1), destination=src,\n                src_nodata=nodata, dst_nodata=0,\n                dst_transform=window.transform, dst_crs=config.CRS,\n                resampling=Resampling.average,\n            )\n\n    if is_int:\n        db = config.dn_to_db(src)\n        db[src <= 0] = 0.0\n    else:\n        valid = src[src != 0]\n        if valid.size == 0:\n            raise ValueError(\"GeoTIFF contains only nodata\")\n        med = float(np.median(valid))\n        if med < 5.0:\n            raise ValueError(\n                f\"float values look like calibrated sigma0 dB (median {med:.1f}); \"\n                \"upload raw GRD DN (integer) or this project's pseudo-dB GeoTIFF\"\n            )\n        db = src\n        db[~np.isfinite(db)] = 0.0\n    if not (db > 0).any():\n        raise ValueError(\"GeoTIFF footprint contains no valid pixels\")\n    return SceneRaster(scene_id=path.stem, db=db.astype(\"float32\"), window=window,\n                       meta={\"source\": \"upload\"})\n\n\ndef save_scene(scene: SceneRaster, path: str | Path) -> Path:\n    \"\"\"Store pseudo-dB as compact uint16 GeoTIFF (dB*100, 0=nodata).\"\"\"\n    from .georef import write_grid_geotiff\n\n    enc = np.clip(np.round(scene.db * DB_SCALE), 0, 65535).astype(\"uint16\")\n    return write_grid_geotiff(path, enc, scene.window, dtype=\"uint16\", nodata=0)\n\n\ndef load_scene(path: str | Path) -> SceneRaster:\n    with rasterio.open(path) as ds:\n        enc = ds.read(1)\n        t = ds.transform\n    col = round((t.c - config.GRID_ORIGIN_LON) / config.PIXEL_DEG)\n    row = round((config.GRID_ORIGIN_LAT - t.f) / config.PIXEL_DEG)\n    window = GridWindow(col, row, enc.shape[1], enc.shape[0])\n    return SceneRaster(Path(path).stem, enc.astype(\"float32\") / DB_SCALE, window)\n\n\ndef to_uint8(db: np.ndarray, clip: tuple[float, float]) -> np.ndarray:\n    from .georef import db_to_uint8\n\n    return db_to_uint8(db, clip)\n", "src/oilspill/dataset_build.py": "\"\"\"Phase 2: build the training dataset from real Cerulean labels + real S1 imagery.\n\nPer scene we store, on the Cerulean grid:\n\n* ``scenes/<scene>.tif``  - uint16 pseudo-dB*100 (0 = nodata)\n* ``masks/<scene>.tif``   - uint8 {0,1,2,3,255} training mask (T1 + T2 labels)\n* ``masks/<scene>_t1.tif`` - uint8 evaluation mask from human-reviewed slicks only\n  (255 everywhere if the scene has no reviewed positive)\n* ``status/<scene>.json`` - done/failed + stats (makes the build resumable)\n\nTraining crops are sampled on the fly from these rasters (see ``train.py``),\nso no tile explosion on disk.\n\nRun:  python -m oilspill.dataset_build --scenes-per-aoi 60 --workers 4\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport logging\nimport random\nimport traceback\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom datetime import UTC, datetime\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import config\nfrom .cerulean_api import CeruleanClient, label_tier, model_class\nfrom .georef import write_grid_geotiff\nfrom .labels import rasterize_slicks, slick_geometry, tier1_mask\nfrom .s1_source import find_item\nfrom .scene import prepare_item, save_scene\n\nlog = logging.getLogger(__name__)\n\nPOS_SQL = \",\".join(str(c) for c in config.POSITIVE_CLS)\nNEG_SQL = \",\".join(str(c) for c in config.HARD_NEGATIVE_CLS)\n\n\ndef data_paths(root: Path) -> dict[str, Path]:\n    p = {\n        \"scenes\": root / \"scenes\",\n        \"masks\": root / \"masks\",\n        \"status\": root / \"status\",\n        \"meta\": root / \"meta\",\n    }\n    for d in p.values():\n        d.mkdir(parents=True, exist_ok=True)\n    return p\n\n\ndef end_date() -> str:\n    return datetime.now(UTC).strftime(\"%Y-%m-%dT00:00:00Z\")\n\n\ndef select_scenes(\n    client: CeruleanClient, scenes_per_aoi: int, start: str = config.DATASET_START_DATE,\n) -> pd.DataFrame:\n    \"\"\"Pick scene ids per AOI: reviewed-positive first, then hard-negative, then T2.\"\"\"\n    dt = f\"{start}T00:00:00Z/{end_date()}\"\n    rows: list[dict] = []\n    for aoi in config.AOIS:\n        seen: dict[str, str] = {}\n        queries = [\n            (\"T1\", f\"hitl_cls IN ({POS_SQL})\", scenes_per_aoi),\n            (\"HN\", f\"hitl_cls IN ({NEG_SQL})\", max(scenes_per_aoi // 5, 3)),\n            (\"T2\", f\"cls IN ({POS_SQL}) AND machine_confidence >= {config.T2_MIN_CONFIDENCE}\",\n             scenes_per_aoi * 15),\n        ]\n        for why, cql, max_feats in queries:\n            if len(seen) >= scenes_per_aoi:\n                break\n            try:\n                feats = list(client.iter_slicks(cql2=cql, bbox=aoi.bbox, datetime_range=dt,\n                                                max_features=max_feats, limit=200))\n            except Exception as exc:  # noqa: BLE001\n                log.warning(\"query %s/%s failed: %s\", aoi.name, why, exc)\n                continue\n            # The API returns rows in id order (i.e. clustered in time); shuffle so the\n            # chosen scenes spread across the whole date range.\n            random.Random(config.SPLIT_SEED).shuffle(feats)\n            for f in feats:\n                sid = f[\"properties\"].get(\"s1_scene_id\")\n                if sid and sid not in seen:\n                    seen[sid] = why\n                    rows.append({\n                        \"scene_id\": sid, \"aoi\": aoi.name, \"reason\": why,\n                        \"slick_timestamp\": f[\"properties\"].get(\"slick_timestamp\"),\n                    })\n                if len(seen) >= scenes_per_aoi:\n                    break\n        log.info(\"AOI %s: %d scenes\", aoi.name, len(seen))\n    return pd.DataFrame(rows)\n\n\ndef build_one(client: CeruleanClient, scene_id: str, aoi: str, paths: dict[str, Path]) -> dict:\n    status_path = paths[\"status\"] / f\"{scene_id}.json\"\n    if status_path.exists():\n        st = json.loads(status_path.read_text())\n        if st.get(\"status\") == \"done\":\n            return st\n\n    try:\n        feats = client.slicks_in_scene(scene_id)\n        item = find_item(scene_id)\n        if item is None:\n            raise RuntimeError(\"no Planetary Computer item\")\n        scene = prepare_item(item)\n        mask, stats = rasterize_slicks(feats, scene.window)\n        mask[~scene.valid] = config.IGNORE_INDEX\n        t1 = tier1_mask(feats, scene.window)\n        t1[~scene.valid] = config.IGNORE_INDEX\n\n        # hard-negative + positive centroids -> crop sampling hints\n        hints = []\n        for f in feats:\n            g = slick_geometry(f)\n            if g is None:\n                continue\n            cls = model_class(f[\"properties\"])\n            tier = label_tier(f[\"properties\"])\n            c = g.representative_point()\n            kind = \"neg\" if cls == 0 else (\"pos\" if cls in config.OIL_CLASSES and tier else \"ign\")\n            hints.append({\"lon\": c.x, \"lat\": c.y, \"kind\": kind, \"cls\": cls, \"tier\": tier})\n\n        pc_id = item[\"id\"]\n        save_scene(scene, paths[\"scenes\"] / f\"{pc_id}.tif\")\n        write_grid_geotiff(paths[\"masks\"] / f\"{pc_id}.tif\", mask, scene.window, dtype=\"uint8\")\n        write_grid_geotiff(paths[\"masks\"] / f\"{pc_id}_t1.tif\", t1, scene.window, dtype=\"uint8\")\n\n        valid = scene.valid\n        b = scene.bounds\n        st = {\n            \"status\": \"done\", \"scene_id\": scene_id, \"pc_id\": pc_id, \"aoi\": aoi,\n            \"datetime\": scene.meta.get(\"datetime\"),\n            \"orbit_direction\": scene.meta.get(\"orbit_direction\"),\n            \"bounds\": b, \"center_lon\": (b[0] + b[2]) / 2, \"center_lat\": (b[1] + b[3]) / 2,\n            \"shape\": list(scene.db.shape), \"valid_px\": int(valid.sum()),\n            \"pos_px\": {str(k): int((mask == k).sum()) for k in config.OIL_CLASSES},\n            \"t1_pos_px\": int(np.isin(t1, config.OIL_CLASSES).sum()),\n            \"n_slicks\": len(feats), \"label_stats\": stats, \"hints\": hints,\n            \"db_p1\": float(np.percentile(scene.db[valid], 1)) if valid.any() else None,\n            \"db_p99\": float(np.percentile(scene.db[valid], 99)) if valid.any() else None,\n            \"read_s\": scene.meta.get(\"read_s\"), \"warp_s\": scene.meta.get(\"warp_s\"),\n        }\n    except Exception as exc:  # noqa: BLE001\n        st = {\"status\": \"failed\", \"scene_id\": scene_id, \"aoi\": aoi,\n              \"error\": f\"{type(exc).__name__}: {exc}\", \"trace\": traceback.format_exc()[-2000:]}\n        log.warning(\"scene %s failed: %s\", scene_id, st[\"error\"])\n    status_path.write_text(json.dumps(st))\n    return st\n\n\ndef make_splits(done: list[dict], seed: int = config.SPLIT_SEED) -> pd.DataFrame:\n    \"\"\"Scene-grouped, spatial (2-degree cell) + temporal split.\n\n    * test  = scenes in the last TEMPORAL_TEST_MONTHS  UNION  scenes in ~15% of cells\n    * val   = scenes in another ~15% of cells\n    * train = the rest\n    A scene belongs to exactly one split (assert below).\n    \"\"\"\n    df = pd.DataFrame([{k: d[k] for k in (\"pc_id\", \"scene_id\", \"aoi\", \"datetime\",\n                                          \"center_lon\", \"center_lat\", \"t1_pos_px\")}\n                       for d in done])\n    df[\"cell\"] = (np.floor(df.center_lon / config.SPLIT_CELL_DEG).astype(int).astype(str) + \"_\"\n                  + np.floor(df.center_lat / config.SPLIT_CELL_DEG).astype(int).astype(str))\n    df[\"acq\"] = pd.to_datetime(df[\"datetime\"], utc=True, errors=\"coerce\")\n    cutoff = df[\"acq\"].max() - pd.DateOffset(months=config.TEMPORAL_TEST_MONTHS)\n\n    rng = np.random.default_rng(seed)\n    cells = np.array(sorted(df.cell.unique()))\n    rng.shuffle(cells)\n    n_test = max(1, round(len(cells) * config.TEST_CELL_FRAC))\n    n_val = max(1, round(len(cells) * config.VAL_CELL_FRAC))\n    test_cells, val_cells = set(cells[:n_test]), set(cells[n_test:n_test + n_val])\n\n    def assign(r) -> str:\n        if r.cell in test_cells or r.acq >= cutoff:\n            return \"test\"\n        if r.cell in val_cells:\n            return \"val\"\n        return \"train\"\n\n    df[\"split\"] = df.apply(assign, axis=1)\n    # Spatial leakage guard: a cell used by test (spatially) must not appear in train/val.\n    assert not (set(df[df.split == \"train\"].cell) & test_cells), \"test cell leaked into train\"\n    assert not (set(df[df.split == \"train\"].cell) & val_cells), \"val cell leaked into train\"\n    assert df.pc_id.is_unique, \"scene appears twice\"\n    return df.drop(columns=[\"acq\"])\n\n\ndef compute_clip(done: list[dict], split_df: pd.DataFrame) -> tuple[float, float]:\n    train_ids = set(split_df[split_df.split == \"train\"].pc_id)\n    p1 = [d[\"db_p1\"] for d in done if d[\"pc_id\"] in train_ids and d.get(\"db_p1\")]\n    p99 = [d[\"db_p99\"] for d in done if d[\"pc_id\"] in train_ids and d.get(\"db_p99\")]\n    return float(np.median(p1)), float(np.median(p99))\n\n\ndef main(argv: list[str] | None = None) -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--root\", default=str(config.DATA_DIR / \"dataset\"))\n    ap.add_argument(\"--scenes-per-aoi\", type=int, default=60)\n    ap.add_argument(\"--workers\", type=int, default=4)\n    ap.add_argument(\"--max-scenes\", type=int, default=None)\n    args = ap.parse_args(argv)\n    logging.basicConfig(level=logging.INFO, format=\"%(asctime)s %(levelname)s %(message)s\")\n\n    root = Path(args.root)\n    paths = data_paths(root)\n    client = CeruleanClient(cache_dir=root / \"cache\" / \"cerulean\")\n\n    sel_path = paths[\"meta\"] / \"selected_scenes.csv\"\n    if sel_path.exists():\n        sel = pd.read_csv(sel_path)\n    else:\n        sel = select_scenes(client, args.scenes_per_aoi)\n        sel.to_csv(sel_path, index=False)\n    if args.max_scenes:\n        sel = sel.head(args.max_scenes)\n    log.info(\"building %d scenes with %d workers\", len(sel), args.workers)\n\n    results = []\n    with ThreadPoolExecutor(max_workers=args.workers) as ex:\n        futs = {ex.submit(build_one, client, r.scene_id, r.aoi, paths): r.scene_id\n                for r in sel.itertuples()}\n        for i, fut in enumerate(as_completed(futs), 1):\n            st = fut.result()\n            results.append(st)\n            if i % 10 == 0 or i == len(futs):\n                n_ok = sum(r[\"status\"] == \"done\" for r in results)\n                log.info(\"progress %d/%d (done=%d)\", i, len(futs), n_ok)\n\n    done = [r for r in results if r[\"status\"] == \"done\"]\n    failed = [r for r in results if r[\"status\"] != \"done\"]\n    # de-duplicate (two Cerulean ids can map to one PC item)\n    uniq = {d[\"pc_id\"]: d for d in done}\n    done = list(uniq.values())\n    if len(done) < 5:\n        log.error(\"only %d scenes built; aborting split\", len(done))\n        return 1\n\n    split_df = make_splits(done)\n    clip = compute_clip(done, split_df)\n    split_df.to_csv(paths[\"meta\"] / \"splits.csv\", index=False)\n    meta = {\n        \"built_at\": datetime.now(UTC).isoformat(),\n        \"n_selected\": int(len(sel)), \"n_done\": len(done), \"n_failed\": len(failed),\n        \"db_clip\": clip,\n        \"split_counts\": split_df.split.value_counts().to_dict(),\n        \"t1_scenes_by_split\": split_df[split_df.t1_pos_px > 0].split.value_counts().to_dict(),\n        \"pos_px_total\": {str(k): int(sum(d[\"pos_px\"][str(k)] for d in done)) for k in config.OIL_CLASSES},\n        \"failures\": [{\"scene_id\": f[\"scene_id\"], \"error\": f.get(\"error\")} for f in failed][:50],\n        \"grid\": {\"pixel_deg\": config.PIXEL_DEG, \"crs\": config.CRS},\n    }\n    (paths[\"meta\"] / \"dataset_meta.json\").write_text(json.dumps(meta, indent=2))\n    log.info(\"dataset meta: %s\", json.dumps({k: meta[k] for k in\n             (\"n_done\", \"n_failed\", \"db_clip\", \"split_counts\", \"t1_scenes_by_split\", \"pos_px_total\")}))\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "src/oilspill/infer.py": "\"\"\"Sliding-window inference over a whole scene.\n\nModel-framework agnostic: a *predictor* is any callable mapping a float32 batch\n``(N, 1, 512, 512)`` to logits ``(N, 4, 512, 512)``. :class:`OnnxPredictor`\nis what the API/batch use; ``train.py`` wraps the torch model the same way, so\nevaluation and serving share this exact code.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport time\nfrom collections.abc import Callable\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom . import config\nfrom .scene import SceneRaster, to_uint8\n\nlog = logging.getLogger(__name__)\n\nPredictor = Callable[[np.ndarray], np.ndarray]\n\n\n@dataclass\nclass ModelCard:\n    model_version: str\n    architecture: str\n    db_clip: tuple[float, float]\n    norm_mean: float\n    norm_std: float\n    classes: dict[str, str]\n    thresholds: dict[str, float]\n    metrics: dict | None = None\n    trained_on: str | None = None\n    raw: dict | None = None\n\n    @classmethod\n    def load(cls, path: str | Path) -> ModelCard:\n        d = json.loads(Path(path).read_text(encoding=\"utf-8\"))\n        return cls(\n            model_version=d[\"model_version\"], architecture=d[\"architecture\"],\n            db_clip=tuple(d[\"db_clip\"]), norm_mean=float(d[\"norm_mean\"]),\n            norm_std=float(d[\"norm_std\"]), classes=d[\"classes\"],\n            thresholds=d.get(\"thresholds\", {}), metrics=d.get(\"metrics\"),\n            trained_on=d.get(\"trained_on\"), raw=d,\n        )\n\n\nclass OnnxPredictor:\n    def __init__(self, onnx_path: str | Path, threads: int | None = None):\n        import onnxruntime as ort\n\n        so = ort.SessionOptions()\n        if threads:\n            so.intra_op_num_threads = threads\n        providers = [p for p in (\"CUDAExecutionProvider\", \"CPUExecutionProvider\")\n                     if p in ort.get_available_providers()]\n        t0 = time.perf_counter()\n        self.session = ort.InferenceSession(str(onnx_path), so, providers=providers)\n        self.load_s = time.perf_counter() - t0\n        self.input_name = self.session.get_inputs()[0].name\n\n    def __call__(self, batch: np.ndarray) -> np.ndarray:\n        return self.session.run(None, {self.input_name: batch.astype(np.float32)})[0]\n\n\ndef _ramp_weights(tile: int, overlap: int) -> np.ndarray:\n    \"\"\"2-D blending weights: 1 in the centre, linear ramp over the overlap.\"\"\"\n    w1 = np.ones(tile, dtype=np.float32)\n    if overlap > 0:\n        ramp = (np.arange(overlap, dtype=np.float32) + 0.5) / overlap\n        w1[:overlap] = ramp\n        w1[-overlap:] = ramp[::-1]\n    return np.outer(w1, w1)\n\n\ndef _starts(n: int, tile: int, stride: int) -> list[int]:\n    if n <= tile:\n        return [0]\n    s = list(range(0, n - tile, stride))\n    s.append(n - tile)\n    return s\n\n\ndef softmax(x: np.ndarray, axis: int = 1) -> np.ndarray:\n    x = x - x.max(axis=axis, keepdims=True)\n    e = np.exp(x)\n    return e / e.sum(axis=axis, keepdims=True)\n\n\ndef normalise(u8: np.ndarray, mean: float, std: float) -> np.ndarray:\n    \"\"\"uint8 grid raster -> model input. Nodata (0) maps to exactly 0.0.\n\n    Shared by training (train.py) and inference so the two cannot diverge.\n    \"\"\"\n    x = ((u8.astype(np.float32) / 255.0) - mean) / std\n    x[u8 == config.NODATA] = 0.0\n    return x\n\n\ndef predict_scene(\n    scene: SceneRaster,\n    predictor: Predictor,\n    card: ModelCard,\n    batch_size: int = 4,\n    tta: bool = False,\n    tile: int = config.TILE_PX,\n    stride: int = config.INFER_STRIDE,\n) -> tuple[np.ndarray, dict[str, float]]:\n    \"\"\"Return per-class probabilities ``(4, H, W)`` float32 and timings.\"\"\"\n    t0 = time.perf_counter()\n    u8 = to_uint8(scene.db, card.db_clip)\n    valid = u8 != config.NODATA\n    H, W = u8.shape\n    pad_h, pad_w = max(0, tile - H), max(0, tile - W)\n    if pad_h or pad_w:\n        u8 = np.pad(u8, ((0, pad_h), (0, pad_w)))\n    Hp, Wp = u8.shape\n    x = normalise(u8, card.norm_mean, card.norm_std)\n\n    acc = np.zeros((config.N_CLASSES, Hp, Wp), dtype=np.float32)\n    wsum = np.zeros((Hp, Wp), dtype=np.float32)\n    wt = _ramp_weights(tile, tile - stride)\n\n    coords = [(r, c) for r in _starts(Hp, tile, stride) for c in _starts(Wp, tile, stride)\n              if (u8[r:r + tile, c:c + tile] != 0).any()]\n    t_pre = time.perf_counter() - t0\n    t1 = time.perf_counter()\n    for i in range(0, len(coords), batch_size):\n        chunk = coords[i:i + batch_size]\n        batch = np.stack([x[r:r + tile, c:c + tile] for r, c in chunk])[:, None]\n        probs = softmax(predictor(batch))\n        if tta:\n            probs = probs + softmax(predictor(batch[..., ::-1].copy()))[..., ::-1]\n            probs = probs + softmax(predictor(batch[..., ::-1, :].copy()))[..., ::-1, :]\n            probs /= 3.0\n        for (r, c), p in zip(chunk, probs, strict=True):\n            acc[:, r:r + tile, c:c + tile] += p * wt\n            wsum[r:r + tile, c:c + tile] += wt\n    t_inf = time.perf_counter() - t1\n\n    wsum[wsum == 0] = 1.0\n    probs = (acc / wsum)[:, :H, :W]\n    # [SRC] Cerulean zeroes non-background probability where the input is nodata.\n    probs[1:, ~valid] = 0.0\n    probs[0, ~valid] = 1.0\n    return probs, {\"preprocess_s\": t_pre, \"inference_s\": t_inf, \"n_tiles\": len(coords)}\n", "src/oilspill/postprocess.py": "\"\"\"Probabilities -> slick polygons, reproducing Cerulean's post-processing [SRC].\n\n1. P(oil) = P(infra) + P(natural) + P(vessel)\n2. islands of P(oil) > p_low (connected components)\n3. keep an island only if max P(oil) >= p_seed\n4. inside kept islands, trim to P(oil) >= p_trim and polygonize\n5. drop polygons > edge_frac within edge_deg of the scene's nodata edge\n6. confidence = MEDIAN P(oil) inside the polygon (mean/max also reported)\n7. class = argmax of per-class probability sums inside the polygon\nPlus our cleanup: make_valid, land buffer removal, min area, simplify.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nfrom dataclasses import asdict\nfrom typing import Any\n\nimport numpy as np\nfrom rasterio.features import rasterize, shapes\nfrom scipy import ndimage\nfrom shapely import make_valid\nfrom shapely.geometry import MultiPolygon, mapping, shape\nfrom shapely.ops import unary_union\n\nfrom . import config\nfrom .crsutil import metric_properties\nfrom .georef import GridWindow\n\nlog = logging.getLogger(__name__)\n\n\ndef _polys(geom) -> list:\n    if geom.is_empty:\n        return []\n    if geom.geom_type == \"Polygon\":\n        return [geom]\n    if geom.geom_type in (\"MultiPolygon\", \"GeometryCollection\"):\n        out = []\n        for g in geom.geoms:\n            out.extend(_polys(g))\n        return out\n    return []\n\n\ndef extract_slicks(\n    probs: np.ndarray,\n    window: GridWindow,\n    valid: np.ndarray,\n    cfg: config.PostprocessConfig = config.POSTPROCESS,\n    land=None,\n    overrides: dict[str, float] | None = None,\n) -> list[dict[str, Any]]:\n    \"\"\"Return GeoJSON-like features (EPSG:4326) with Cerulean-style properties.\"\"\"\n    c = asdict(cfg)\n    c.update(overrides or {})\n    p_oil = probs[1:].sum(axis=0)\n    p_oil[~valid] = 0.0\n\n    islands, n = ndimage.label(p_oil > c[\"p_low\"], structure=np.ones((3, 3)))\n    if n == 0:\n        return []\n    maxes = ndimage.maximum(p_oil, islands, index=np.arange(1, n + 1))\n    keep_ids = np.nonzero(np.asarray(maxes) >= c[\"p_seed\"])[0] + 1\n    if keep_ids.size == 0:\n        return []\n    kept = np.isin(islands, keep_ids)\n    trimmed = kept & (p_oil >= c[\"p_trim\"])\n    if not trimmed.any():\n        return []\n\n    # Polygonize each connected trimmed region inside its own bounding slice\n    # (never full-raster work per region - a scene can have thousands of specks).\n    regions, nreg = ndimage.label(trimmed, structure=np.ones((3, 3)))\n    edge_zone = _edge_zone(valid, c[\"edge_deg\"])\n    px = config.PIXEL_DEG\n    lat_mid = 0.5 * (window.bounds[1] + window.bounds[3])\n    px_area_m2 = (px * 111_320.0) ** 2 * max(np.cos(np.radians(lat_mid)), 0.01)\n    min_px = max(1, int(0.5 * c[\"min_area_m2\"] / px_area_m2))  # cheap pre-filter; exact check below\n    sizes = ndimage.sum_labels(np.ones_like(regions, dtype=np.int32), regions, index=np.arange(1, nreg + 1))\n    slices = ndimage.find_objects(regions)\n    land_mask = _burn(land, window) if land is not None else None\n    t = window.transform\n    feats: list[dict[str, Any]] = []\n    for rid0, sl in enumerate(slices):\n        rid = rid0 + 1\n        if sl is None or sizes[rid0] < min_px:\n            continue\n        region = regions[sl] == rid\n        # edge rule [SRC]\n        if (edge_zone[sl] & region).sum() > c[\"edge_frac\"] * region.sum():\n            continue\n        if land_mask is not None:\n            region = region & ~land_mask[sl]\n            if region.sum() < min_px:\n                continue\n        # fill pinholes (< max_hole_px) so speckle does not explode the vertex count\n        holes = ndimage.binary_fill_holes(region) & ~region\n        if holes.any():\n            hl, nh = ndimage.label(holes)\n            hs = ndimage.sum_labels(holes, hl, index=np.arange(1, nh + 1))\n            small = np.nonzero(hs < c.get(\"max_hole_px\", 16))[0] + 1\n            region = region | np.isin(hl, small)\n        sub_t = t @ t.translation(sl[1].start, sl[0].start)\n        polys = []\n        for gj, val in shapes(region.astype(np.uint8), mask=region, transform=sub_t, connectivity=8):\n            if val:\n                g = shape(gj)\n                polys.extend(_polys(g if g.is_valid else make_valid(g)))\n        if not polys:\n            continue\n        geom = polys[0] if len(polys) == 1 else unary_union(polys)\n        geom = geom.simplify(c[\"simplify_px\"] * px, preserve_topology=True)\n        if geom.is_empty:\n            continue\n        if geom.geom_type == \"Polygon\":\n            geom = MultiPolygon([geom])\n        elif geom.geom_type != \"MultiPolygon\":\n            geom = MultiPolygon(_polys(geom))\n            if geom.is_empty:\n                continue\n        m = metric_properties(geom)\n        if m[\"area_m2\"] < c[\"min_area_m2\"]:\n            continue\n\n        vals = p_oil[sl][region]\n        cls_sums = probs[1:, sl[0], sl[1]][:, region].sum(axis=1)\n        cls = int(np.argmax(cls_sums)) + 1\n        cen = geom.centroid\n        feats.append({\n            \"type\": \"Feature\",\n            \"geometry\": mapping(geom),\n            \"properties\": {\n                \"cls\": cls,\n                \"cls_name\": config.MODEL_CLASSES[cls],\n                \"confidence\": float(np.median(vals)),\n                \"confidence_mean\": float(vals.mean()),\n                \"confidence_max\": float(vals.max()),\n                \"class_prob_share\": [float(v) for v in cls_sums / max(cls_sums.sum(), 1e-9)],\n                \"area_km2\": m[\"area_m2\"] / 1e6,\n                \"perimeter_km\": m[\"perimeter_m\"] / 1e3,\n                \"length_km\": m[\"length_m\"] / 1e3,\n                \"polsby_popper\": m[\"polsby_popper\"],\n                \"centroid_lon\": cen.x,\n                \"centroid_lat\": cen.y,\n                \"bbox\": list(geom.bounds),\n                \"n_pixels\": int(region.sum()),\n            },\n        })\n    feats.sort(key=lambda f: -f[\"properties\"][\"confidence\"])\n    return feats\n\n\ndef _edge_zone(valid: np.ndarray, edge_deg: float) -> np.ndarray:\n    \"\"\"Pixels within edge_deg of nodata (the swath edge) or the raster border.\"\"\"\n    k = max(1, int(round(edge_deg / config.PIXEL_DEG)))\n    invalid = ~valid\n    invalid = np.pad(invalid, 1, constant_values=True)\n    grown = ndimage.binary_dilation(invalid, iterations=k)[1:-1, 1:-1]\n    return grown & valid\n\n\ndef _burn(geom, window: GridWindow) -> np.ndarray:\n    return rasterize([(geom, 1)], out_shape=window.shape, transform=window.transform,\n                     fill=0, dtype=\"uint8\").astype(bool)\n\n\ndef to_feature_collection(features: list[dict[str, Any]], **props) -> dict[str, Any]:\n    return {\"type\": \"FeatureCollection\", \"features\": features, **props}\n", "src/oilspill/metrics.py": "\"\"\"Segmentation metrics (numpy only, shared by training and evaluation).\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom . import config\n\n\nclass ConfMat:\n    def __init__(self, n: int = config.N_CLASSES):\n        self.m = np.zeros((n, n), dtype=np.int64)\n\n    def update(self, pred: np.ndarray, target: np.ndarray):\n        v = target != config.IGNORE_INDEX\n        self.m += np.bincount(target[v].astype(np.int64) * self.m.shape[0] + pred[v].astype(np.int64),\n                              minlength=self.m.size).reshape(self.m.shape)\n\n    def summary(self) -> dict:\n        m = self.m.astype(np.float64)\n        out = {}\n        for c in range(1, m.shape[0]):\n            tp, fp, fn = m[c, c], m[:, c].sum() - m[c, c], m[c, :].sum() - m[c, c]\n            out[config.MODEL_CLASSES[c]] = _prf(tp, fp, fn)\n        tp = m[1:, 1:].sum()\n        fp = m[0, 1:].sum()\n        fn = m[1:, 0].sum()\n        out[\"OIL_binary\"] = _prf(tp, fp, fn)\n        out[\"confusion_matrix\"] = self.m.tolist()\n        return out\n\n\ndef _prf(tp, fp, fn) -> dict:\n    p = tp / (tp + fp) if tp + fp else 0.0\n    r = tp / (tp + fn) if tp + fn else 0.0\n    f = 2 * p * r / (p + r) if p + r else 0.0\n    iou = tp / (tp + fp + fn) if tp + fp + fn else 0.0\n    return {\"precision\": p, \"recall\": r, \"f1_dice\": f, \"iou\": iou, \"tp\": int(tp), \"fp\": int(fp), \"fn\": int(fn)}\n", "src/oilspill/train.py": "\"\"\"Phase 4/5: train the U-Net (ResNet34, 1-channel VV, 4 classes) on the built dataset.\n\nCrops are sampled on the fly from whole-scene rasters:\n  50% centred on a labelled oil pixel, 20% on a hard-negative (human-rejected\n  look-alike), 30% anywhere valid. Validation uses a FIXED list of crops from\n  the val scenes, so val numbers are comparable epoch to epoch.\n\nResumable: ``last.pt`` (model/optimizer/scaler/scheduler/epoch/best/RNG) is\nwritten every epoch; ``--resume`` continues after a Colab disconnect.\n\nRun (Colab):  python -m oilspill.train --data /content/data/dataset --out /content/drive/MyDrive/oilspill/run1\nOverfit test: python -m oilspill.train ... --overfit 20 --epochs 60\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport logging\nimport math\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport rasterio\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom . import config\nfrom .georef import db_to_uint8\nfrom .infer import normalise\nfrom .metrics import ConfMat\n\nlog = logging.getLogger(__name__)\nT = config.TILE_PX\n\n\n# ---------------------------------------------------------------------------\n# Data\n# ---------------------------------------------------------------------------\n\nclass SceneStore:\n    \"\"\"All scenes of one split held in RAM as uint8 (image) + uint8 (mask).\"\"\"\n\n    def __init__(self, root: Path, ids: list[str], clip: tuple[float, float], mask_suffix: str = \"\"):\n        self.images, self.masks, self.ids = [], [], []\n        self.pos_idx, self.neg_pts = [], []\n        for pid in ids:\n            sp, mp = root / \"scenes\" / f\"{pid}.tif\", root / \"masks\" / f\"{pid}{mask_suffix}.tif\"\n            if not (sp.exists() and mp.exists()):\n                continue\n            with rasterio.open(sp) as d:\n                db = d.read(1).astype(np.float32) / 100.0\n            with rasterio.open(mp) as d:\n                m = d.read(1)\n            img = db_to_uint8(db, clip)\n            self.images.append(img)\n            self.masks.append(m)\n            self.ids.append(pid)\n            pos = np.flatnonzero(np.isin(m, config.OIL_CLASSES))\n            if pos.size > 20000:\n                pos = np.random.default_rng(0).choice(pos, 20000, replace=False)\n            self.pos_idx.append(pos)\n            st = json.loads((root / \"status_by_pc\" / f\"{pid}.json\").read_text()) \\\n                if (root / \"status_by_pc\" / f\"{pid}.json\").exists() else {}\n            self.neg_pts.append(st.get(\"neg_px\", []))\n        log.info(\"loaded %d scenes (%.1f GB)\", len(self.ids),\n                 sum(i.nbytes + m.nbytes for i, m in zip(self.images, self.masks, strict=True)) / 1e9)\n\n    def crop(self, k: int, cy: int, cx: int) -> tuple[np.ndarray, np.ndarray]:\n        img, m = self.images[k], self.masks[k]\n        H, W = img.shape\n        y0 = int(np.clip(cy - T // 2, 0, max(H - T, 0)))\n        x0 = int(np.clip(cx - T // 2, 0, max(W - T, 0)))\n        ci = img[y0:y0 + T, x0:x0 + T]\n        cm = m[y0:y0 + T, x0:x0 + T]\n        if ci.shape != (T, T):\n            pi = np.zeros((T, T), np.uint8)\n            pm = np.full((T, T), config.IGNORE_INDEX, np.uint8)\n            pi[:ci.shape[0], :ci.shape[1]] = ci\n            pm[:cm.shape[0], :cm.shape[1]] = cm\n            ci, cm = pi, pm\n        return ci, cm\n\n\ndef augment(img: np.ndarray, m: np.ndarray, rng: random.Random) -> tuple[np.ndarray, np.ndarray]:\n    k = rng.randrange(4)\n    img, m = np.rot90(img, k), np.rot90(m, k)\n    if rng.random() < 0.5:\n        img, m = img[:, ::-1], m[:, ::-1]\n    img = np.ascontiguousarray(img)\n    m = np.ascontiguousarray(m)\n    valid = img > 0\n    if rng.random() < 0.8:  # brightness/contrast jitter (+-10%) on valid pixels only\n        f = img.astype(np.float32)\n        a, b = rng.uniform(0.9, 1.1), rng.uniform(-12, 12)\n        f = (f - 128) * a + 128 + b\n        if rng.random() < 0.3:\n            f += np.random.default_rng(rng.randrange(1 << 30)).normal(0, 3, f.shape)\n        img = np.where(valid, np.clip(np.round(f), 1, 255), 0).astype(np.uint8)\n    return img, m\n\n\nclass CropDataset(Dataset):\n    def __init__(self, store: SceneStore, n: int, mean: float, std: float, seed: int = 0,\n                 fixed: list[tuple[int, int, int]] | None = None, aug: bool = True):\n        self.s, self.n, self.mean, self.std = store, n, mean, std\n        self.seed, self.fixed, self.aug = seed, fixed, aug\n        self.epoch = 0\n        w = np.array([max(len(p), 1) ** 0.5 for p in store.pos_idx], dtype=np.float64)\n        self.scene_w = w / w.sum()\n\n    def __len__(self) -> int:\n        return len(self.fixed) if self.fixed is not None else self.n\n\n    def _sample(self, rng: random.Random) -> tuple[int, int, int]:\n        s = self.s\n        k = int(np.random.default_rng(rng.randrange(1 << 30)).choice(len(s.ids), p=self.scene_w))\n        H, W = s.images[k].shape\n        u = rng.random()\n        if u < 0.5 and len(s.pos_idx[k]):\n            flat = int(s.pos_idx[k][rng.randrange(len(s.pos_idx[k]))])\n            cy, cx = divmod(flat, W)\n            cy += rng.randint(-T // 3, T // 3)\n            cx += rng.randint(-T // 3, T // 3)\n        elif u < 0.7 and s.neg_pts[k]:\n            cy, cx = s.neg_pts[k][rng.randrange(len(s.neg_pts[k]))]\n            cy += rng.randint(-T // 4, T // 4)\n            cx += rng.randint(-T // 4, T // 4)\n        else:\n            for _ in range(20):\n                cy, cx = rng.randrange(H), rng.randrange(W)\n                if s.images[k][cy, cx] > 0:\n                    break\n        return k, cy, cx\n\n    def __getitem__(self, i: int):\n        if self.fixed is not None:\n            k, cy, cx = self.fixed[i]\n            rng = random.Random(i)\n        else:\n            rng = random.Random(hash((self.seed, self.epoch, i)) & 0xFFFFFFFF)\n            k, cy, cx = self._sample(rng)\n        img, m = self.s.crop(k, cy, cx)\n        if self.aug:\n            img, m = augment(img, m, rng)\n        x = normalise(img, self.mean, self.std)[None]\n        return torch.from_numpy(x), torch.from_numpy(m.astype(np.int64))\n\n\ndef fixed_val_crops(store: SceneStore, max_crops: int = 400, seed: int = 1) -> list[tuple[int, int, int]]:\n    rng = np.random.default_rng(seed)\n    pos, neg = [], []\n    for k, (img, m) in enumerate(zip(store.images, store.masks, strict=True)):\n        H, W = img.shape\n        for y in range(0, max(H - T, 0) + 1, T):\n            for x in range(0, max(W - T, 0) + 1, T):\n                ti, tm = img[y:y + T, x:x + T], m[y:y + T, x:x + T]\n                if (ti > 0).mean() < 0.5:\n                    continue\n                c = (k, y + T // 2, x + T // 2)\n                (pos if np.isin(tm, config.OIL_CLASSES).any() else neg).append(c)\n    rng.shuffle(pos)\n    rng.shuffle(neg)\n    n_pos = min(len(pos), max_crops // 2)\n    return pos[:n_pos] + neg[:max_crops - n_pos]\n\n\n# ---------------------------------------------------------------------------\n# Model / loss / metrics\n# ---------------------------------------------------------------------------\n\ndef build_model(encoder: str = \"resnet34\", weights: str | None = \"imagenet\", arch: str = \"Unet\"):\n    import segmentation_models_pytorch as smp\n\n    cls = getattr(smp, arch)\n    return cls(encoder_name=encoder, encoder_weights=weights, in_channels=1, classes=config.N_CLASSES)\n\n\ndef dice_loss(logits: torch.Tensor, target: torch.Tensor, classes=(1, 2, 3), eps: float = 1.0):\n    valid = (target != config.IGNORE_INDEX)\n    probs = logits.float().softmax(1)\n    t = target.clone()\n    t[~valid] = 0\n    onehot = F.one_hot(t, config.N_CLASSES).permute(0, 3, 1, 2).float()\n    v = valid.unsqueeze(1).float()\n    losses = []\n    for c in classes:\n        p, g = probs[:, c] * v[:, 0], onehot[:, c] * v[:, 0]\n        inter = (p * g).sum()\n        losses.append(1 - (2 * inter + eps) / (p.sum() + g.sum() + eps))\n    # binary oil dice as well: the product metric we care most about\n    p_oil = probs[:, 1:].sum(1) * v[:, 0]\n    g_oil = (onehot[:, 1:].sum(1)) * v[:, 0]\n    losses.append(1 - (2 * (p_oil * g_oil).sum() + eps) / (p_oil.sum() + g_oil.sum() + eps))\n    return torch.stack(losses).mean()\n\n\n# ---------------------------------------------------------------------------\n# Training loop\n# ---------------------------------------------------------------------------\n\ndef load_split(data: Path) -> tuple[pd.DataFrame, dict]:\n    meta = json.loads((data / \"meta\" / \"dataset_meta.json\").read_text())\n    splits = pd.read_csv(data / \"meta\" / \"splits.csv\")\n    return splits, meta\n\n\ndef index_status(data: Path) -> None:\n    \"\"\"Map status/<cerulean id>.json -> status_by_pc/<pc id>.json with hard-negative pixel coords.\"\"\"\n    out = data / \"status_by_pc\"\n    out.mkdir(exist_ok=True)\n    for p in (data / \"status\").glob(\"*.json\"):\n        st = json.loads(p.read_text())\n        if st.get(\"status\") != \"done\":\n            continue\n        b = st[\"bounds\"]\n        # bounds are the scene window outer edges -> pixel coords\n        negs = []\n        for h in st.get(\"hints\", []):\n            if h[\"kind\"] == \"neg\":\n                negs.append([int((b[3] - h[\"lat\"]) / config.PIXEL_DEG), int((h[\"lon\"] - b[0]) / config.PIXEL_DEG)])\n        prev = out / f\"{st['pc_id']}.json\"\n        if prev.exists():\n            negs += json.loads(prev.read_text()).get(\"neg_px\", [])\n        prev.write_text(json.dumps({\"neg_px\": negs}))\n\n\ndef compute_norm(store: SceneStore) -> tuple[float, float]:\n    vals = np.concatenate([i[i > 0][:: max(1, (i > 0).sum() // 200000)] for i in store.images])\n    f = vals.astype(np.float64) / 255.0\n    return float(f.mean()), float(f.std())\n\n\ndef class_weights(store: SceneStore) -> torch.Tensor:\n    counts = np.zeros(config.N_CLASSES, dtype=np.float64)\n    for m in store.masks:\n        bc = np.bincount(m[m != config.IGNORE_INDEX].ravel(), minlength=config.N_CLASSES)[:config.N_CLASSES]\n        counts += bc\n    freq = counts / counts.sum()\n    w = 1.0 / np.sqrt(np.maximum(freq, 1e-6))\n    w = w / w[0]\n    w = np.minimum(w, 20.0)\n    log.info(\"class pixel counts %s -> CE weights %s\", counts.astype(int).tolist(), np.round(w, 2).tolist())\n    return torch.tensor(w, dtype=torch.float32)\n\n\n@torch.no_grad()\ndef validate(model, loader, device, amp: bool) -> dict:\n    model.eval()\n    cm = ConfMat()\n    for x, y in loader:\n        x = x.to(device, non_blocking=True)\n        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):\n            logits = model(x)\n        cm.update(logits.argmax(1).cpu().numpy(), y.numpy())\n    return cm.summary()\n\n\ndef main(argv: list[str] | None = None) -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--data\", required=True)\n    ap.add_argument(\"--out\", required=True)\n    ap.add_argument(\"--arch\", default=\"Unet\")\n    ap.add_argument(\"--encoder\", default=\"resnet34\")\n    ap.add_argument(\"--epochs\", type=int, default=40)\n    ap.add_argument(\"--samples-per-epoch\", type=int, default=2400)\n    ap.add_argument(\"--batch-size\", type=int, default=8)\n    ap.add_argument(\"--accum\", type=int, default=1)\n    ap.add_argument(\"--lr\", type=float, default=3e-4)\n    ap.add_argument(\"--wd\", type=float, default=1e-4)\n    ap.add_argument(\"--patience\", type=int, default=8)\n    ap.add_argument(\"--workers\", type=int, default=2)\n    ap.add_argument(\"--resume\", action=\"store_true\")\n    ap.add_argument(\"--overfit\", type=int, default=0, help=\"overfit N fixed train crops (sanity gate)\")\n    ap.add_argument(\"--max-hours\", type=float, default=10.0)\n    args = ap.parse_args(argv)\n    logging.basicConfig(level=logging.INFO, format=\"%(asctime)s %(levelname)s %(message)s\")\n\n    data, out = Path(args.data), Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n    torch.manual_seed(0)\n    np.random.seed(0)\n    random.seed(0)\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    if device.type != \"cuda\":\n        log.warning(\"NO GPU - training on CPU will be extremely slow\")\n    amp = device.type == \"cuda\"\n\n    index_status(data)\n    splits, meta = load_split(data)\n    clip = tuple(meta[\"db_clip\"])\n    train_ids = splits[splits.split == \"train\"].pc_id.tolist()\n    val_ids = splits[splits.split == \"val\"].pc_id.tolist()\n    assert not set(train_ids) & set(val_ids)\n    train_store = SceneStore(data, train_ids, clip)\n    val_store = SceneStore(data, val_ids, clip)\n    mean, std = compute_norm(train_store)\n    cw = class_weights(train_store).to(device)\n\n    if args.overfit:\n        fixed = fixed_val_crops(train_store, args.overfit, seed=3)\n        train_ds = CropDataset(train_store, 0, mean, std, fixed=fixed, aug=False)\n        val_ds = CropDataset(train_store, 0, mean, std, fixed=fixed, aug=False)\n    else:\n        train_ds = CropDataset(train_store, args.samples_per_epoch, mean, std, seed=0)\n        val_ds = CropDataset(val_store, 0, mean, std, fixed=fixed_val_crops(val_store), aug=False)\n    log.info(\"train crops/epoch=%d  val crops=%d  mean=%.4f std=%.4f clip=%s\",\n             len(train_ds), len(val_ds), mean, std, clip)\n\n    train_dl = DataLoader(train_ds, batch_size=args.batch_size, shuffle=bool(args.overfit),\n                          num_workers=args.workers, pin_memory=amp, drop_last=not args.overfit,\n                          persistent_workers=False)  # workers must see the new epoch's sampling seed\n    val_dl = DataLoader(val_ds, batch_size=args.batch_size, num_workers=args.workers, pin_memory=amp)\n\n    model = build_model(args.encoder, \"imagenet\", args.arch).to(device)\n    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)\n    steps_per_epoch = math.ceil(len(train_dl) / args.accum)\n    total = steps_per_epoch * args.epochs\n    warm = steps_per_epoch\n\n    def lr_lambda(step):\n        if step < warm:\n            return (step + 1) / warm\n        return 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warm) / max(1, total - warm))))\n\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=amp)\n\n    run_cfg = {\"arch\": args.arch, \"encoder\": args.encoder, \"db_clip\": clip, \"norm_mean\": mean,\n               \"norm_std\": std, \"class_weights\": cw.tolist(), \"args\": vars(args),\n               \"n_train_scenes\": len(train_store.ids), \"n_val_scenes\": len(val_store.ids)}\n    (out / \"run_config.json\").write_text(json.dumps(run_cfg, indent=2))\n\n    start_epoch, best, bad = 0, -1.0, 0\n    last_path, best_path = out / \"last.pt\", out / \"best.pt\"\n    if args.resume and last_path.exists():\n        ck = torch.load(last_path, map_location=device, weights_only=False)\n        model.load_state_dict(ck[\"model\"])\n        opt.load_state_dict(ck[\"opt\"])\n        sched.load_state_dict(ck[\"sched\"])\n        scaler.load_state_dict(ck[\"scaler\"])\n        start_epoch, best, bad = ck[\"epoch\"] + 1, ck[\"best\"], ck[\"bad\"]\n        random.setstate(ck[\"py_rng\"])\n        np.random.set_state(ck[\"np_rng\"])\n        torch.set_rng_state(ck[\"torch_rng\"].cpu())  # map_location moved it to cuda\n        log.info(\"resumed from epoch %d (best oil dice %.4f)\", ck[\"epoch\"], best)\n\n    log_path = out / \"log.csv\"\n    t_start = time.time()\n    for epoch in range(start_epoch, args.epochs):\n        train_ds.epoch = epoch\n        model.train()\n        t0, run_loss, n = time.time(), 0.0, 0\n        opt.zero_grad(set_to_none=True)\n        for i, (x, y) in enumerate(train_dl):\n            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):\n                logits = model(x)\n                ce = F.cross_entropy(logits.float(), y, weight=cw, ignore_index=config.IGNORE_INDEX)\n                loss = ce + dice_loss(logits, y)\n            if not torch.isfinite(loss):\n                log.error(\"non-finite loss at epoch %d step %d; skipping batch\", epoch, i)\n                opt.zero_grad(set_to_none=True)\n                continue\n            scaler.scale(loss / args.accum).backward()\n            if (i + 1) % args.accum == 0:\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                scaler.step(opt)\n                scaler.update()\n                opt.zero_grad(set_to_none=True)\n                sched.step()\n            run_loss += loss.item()\n            n += 1\n        train_s = time.time() - t0\n        vm = validate(model, val_dl, device, amp)\n        oil = vm[\"OIL_binary\"][\"f1_dice\"]\n        vram = torch.cuda.max_memory_allocated() / 1e9 if amp else 0.0\n        row = {\"epoch\": epoch, \"loss\": run_loss / max(n, 1), \"val_oil_dice\": oil,\n               \"val_oil_iou\": vm[\"OIL_binary\"][\"iou\"],\n               \"val_infra_f1\": vm[\"INFRA\"][\"f1_dice\"], \"val_natural_f1\": vm[\"NATURAL\"][\"f1_dice\"],\n               \"val_vessel_f1\": vm[\"VESSEL\"][\"f1_dice\"], \"lr\": opt.param_groups[0][\"lr\"],\n               \"epoch_s\": round(train_s, 1), \"vram_gb\": round(vram, 2)}\n        new = not log_path.exists()\n        with log_path.open(\"a\", newline=\"\") as fh:\n            w = csv.DictWriter(fh, fieldnames=list(row))\n            if new:\n                w.writeheader()\n            w.writerow(row)\n        log.info(\"epoch %d loss %.4f val_oil_dice %.4f (infra %.3f nat %.3f vessel %.3f) %.0fs vram %.1fGB\",\n                 epoch, row[\"loss\"], oil, row[\"val_infra_f1\"], row[\"val_natural_f1\"],\n                 row[\"val_vessel_f1\"], train_s, vram)\n\n        if oil > best:\n            best, bad = oil, 0\n            torch.save({\"model\": model.state_dict(), \"epoch\": epoch, \"val\": vm, **run_cfg}, best_path)\n            (out / \"best_val_metrics.json\").write_text(json.dumps({\"epoch\": epoch, **vm}, indent=2))\n        else:\n            bad += 1\n        torch.save({\"model\": model.state_dict(), \"opt\": opt.state_dict(), \"sched\": sched.state_dict(),\n                    \"scaler\": scaler.state_dict(), \"epoch\": epoch, \"best\": best, \"bad\": bad,\n                    \"py_rng\": random.getstate(), \"np_rng\": np.random.get_state(),\n                    \"torch_rng\": torch.get_rng_state()}, last_path)\n        if bad >= args.patience and not args.overfit:\n            log.info(\"early stopping at epoch %d (best %.4f)\", epoch, best)\n            break\n        if (time.time() - t_start) / 3600 > args.max_hours:\n            log.info(\"time budget reached; stop (resume with --resume)\")\n            break\n    log.info(\"done. best val oil dice = %.4f  -> %s\", best, best_path)\n    if args.overfit:\n        ok = best > 0.9\n        log.info(\"OVERFIT GATE %s (need > 0.9)\", \"PASSED\" if ok else \"FAILED\")\n        return 0 if ok else 2\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "src/oilspill/export.py": "\"\"\"Phase 7: best.pt -> model.onnx + model_card.json (the artifact the API serves).\n\nVerifies ONNX == torch (max abs diff on real validation crops) before writing\nthe card, so a broken export can never be shipped silently.\n\nRun: python -m oilspill.export --run /content/drive/MyDrive/oilspill/run1 --data /content/data/dataset\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport logging\nimport subprocess\nfrom datetime import UTC, datetime\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom . import config\nfrom .train import build_model\n\nlog = logging.getLogger(__name__)\n\n\ndef git_hash() -> str:\n    try:\n        return subprocess.check_output([\"git\", \"rev-parse\", \"--short\", \"HEAD\"], text=True).strip()\n    except Exception:  # noqa: BLE001\n        return \"nogit\"\n\n\ndef export(run: Path, out: Path, test_batch: np.ndarray | None = None, metrics: dict | None = None) -> dict:\n    ck = torch.load(run / \"best.pt\", map_location=\"cpu\", weights_only=False)\n    model = build_model(ck[\"encoder\"], None, ck[\"arch\"])\n    model.load_state_dict(ck[\"model\"])\n    model.eval()\n\n    out.mkdir(parents=True, exist_ok=True)\n    onnx_path = out / \"model.onnx\"\n    dummy = torch.zeros(1, 1, config.TILE_PX, config.TILE_PX)\n    torch.onnx.export(\n        model, dummy, str(onnx_path), input_names=[\"input\"], output_names=[\"logits\"],\n        dynamic_axes={\"input\": {0: \"batch\"}, \"logits\": {0: \"batch\"}}, opset_version=17,\n        dynamo=False,\n    )\n\n    import onnxruntime as ort\n\n    sess = ort.InferenceSession(str(onnx_path), providers=[\"CPUExecutionProvider\"])\n    x = test_batch if test_batch is not None else np.random.default_rng(0).normal(size=(2, 1, 512, 512))\n    x = x.astype(np.float32)\n    with torch.no_grad():\n        ref = model(torch.from_numpy(x)).numpy()\n    got = sess.run(None, {\"input\": x})[0]\n    diff = float(np.abs(ref - got).max())\n    log.info(\"ONNX vs torch max abs diff = %.2e\", diff)\n    if diff > 1e-3:\n        raise RuntimeError(f\"ONNX export mismatch: {diff}\")\n\n    version = datetime.now(UTC).strftime(\"v%Y%m%d-%H%M\") + f\"-{git_hash()}\"\n    card = {\n        \"model_version\": version,\n        \"architecture\": f\"{ck['arch']}-{ck['encoder']} (segmentation_models_pytorch), 1-channel VV, 4 classes\",\n        \"input\": {\"tile_px\": config.TILE_PX, \"pixel_deg\": config.PIXEL_DEG, \"crs\": config.CRS,\n                  \"radiometry\": \"pseudo-dB = 10*log10(DN^2+1) of uncalibrated S1 GRD VV, clipped to db_clip -> uint8 1..255 (0 = nodata) -> /255 -> (x-mean)/std, nodata -> 0\"},\n        \"db_clip\": list(ck[\"db_clip\"]),\n        \"norm_mean\": ck[\"norm_mean\"],\n        \"norm_std\": ck[\"norm_std\"],\n        \"classes\": {str(k): v for k, v in config.MODEL_CLASSES.items() if k},\n        \"thresholds\": {\"p_low\": config.POSTPROCESS.p_low, \"p_seed\": config.POSTPROCESS.p_seed,\n                       \"p_trim\": config.POSTPROCESS.p_trim, \"min_area_m2\": config.POSTPROCESS.min_area_m2},\n        \"best_epoch\": ck[\"epoch\"],\n        \"val_metrics_crops\": ck.get(\"val\"),\n        \"metrics\": metrics,\n        \"onnx_torch_max_abs_diff\": diff,\n        \"trained_on\": f\"{ck.get('n_train_scenes')} Sentinel-1 scenes; labels derived from SkyTruth Cerulean API (CC BY-SA 4.0)\",\n        \"license\": \"weights CC BY-SA 4.0 (derived from Cerulean data); code Apache-2.0; non-commercial conservation use per SkyTruth ToS\",\n        \"attribution\": [config.ATTRIBUTION_CERULEAN, config.ATTRIBUTION_COPERNICUS],\n        \"exported_at\": datetime.now(UTC).isoformat(),\n    }\n    (out / \"model_card.json\").write_text(json.dumps(card, indent=2))\n    log.info(\"wrote %s and model_card.json (%s)\", onnx_path, version)\n    return card\n\n\ndef main(argv=None) -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--run\", required=True)\n    ap.add_argument(\"--out\", default=None)\n    ap.add_argument(\"--metrics\", default=None, help=\"path to test metrics json to embed\")\n    args = ap.parse_args(argv)\n    logging.basicConfig(level=logging.INFO)\n    run = Path(args.run)\n    metrics = json.loads(Path(args.metrics).read_text()) if args.metrics else None\n    export(run, Path(args.out) if args.out else run / \"export\", metrics=metrics)\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "src/oilspill/evaluate.py": "\"\"\"Phase 6: full-scene evaluation with the exact production pipeline.\n\n* thresholds (p_seed, p_trim) are tuned on VAL scenes only\n* TEST is evaluated once, against human-reviewed (Tier-1) masks only\n  (unreviewed machine labels are ignore, never counted as hit or miss)\n* pixel metrics (binary oil + per class), instance metrics (polygon IoU >= 0.1),\n  confusion matrix\n\nWorks with either the torch checkpoint (Colab) or the exported ONNX model.\n\nRun: python -m oilspill.evaluate --data DATA --model RUN/export  (ONNX)\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport logging\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom rasterio.features import rasterize\nfrom shapely.geometry import shape\n\nfrom . import config\nfrom .infer import ModelCard, OnnxPredictor, predict_scene\nfrom .postprocess import extract_slicks\nfrom .scene import load_scene\nfrom .metrics import ConfMat, _prf\n\nlog = logging.getLogger(__name__)\n\n\ndef _read_mask(path: Path) -> np.ndarray:\n    import rasterio\n\n    with rasterio.open(path) as d:\n        return d.read(1)\n\n\ndef instance_scores(feats: list[dict], gt_mask: np.ndarray, window, iou_thr: float) -> tuple[int, int, int]:\n    \"\"\"Match predicted polygons to GT connected components by IoU (pixel space).\"\"\"\n    from scipy import ndimage\n\n    gt_oil = np.isin(gt_mask, config.OIL_CLASSES)\n    ignore = gt_mask == config.IGNORE_INDEX\n    lab, n = ndimage.label(gt_oil, structure=np.ones((3, 3)))\n    matched_gt = set()\n    tp = fp = 0\n    for f in feats:\n        pm = rasterize([(shape(f[\"geometry\"]), 1)], out_shape=gt_mask.shape,\n                       transform=window.transform, fill=0, dtype=\"uint8\").astype(bool)\n        if (pm & ignore).sum() > 0.5 * pm.sum():\n            continue  # prediction lies on unreviewed/ignored area -> not scored\n        ids = np.unique(lab[pm])\n        ids = ids[ids > 0]\n        best = 0.0\n        best_id = None\n        for i in ids:\n            g = lab == i\n            iou = (g & pm).sum() / (g | pm).sum()\n            if iou > best:\n                best, best_id = iou, i\n        if best >= iou_thr:\n            tp += 1\n            matched_gt.add(best_id)\n        else:\n            fp += 1\n    fn = n - len(matched_gt)\n    return tp, fp, fn\n\n\nclass TorchPredictor:\n    \"\"\"Wrap best.pt for GPU evaluation on Colab (same predictor interface as ONNX).\"\"\"\n\n    def __init__(self, ckpt: Path):\n        import torch\n\n        from .train import build_model\n\n        ck = torch.load(ckpt, map_location=\"cpu\", weights_only=False)\n        self.torch = torch\n        self.dev = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n        self.model = build_model(ck[\"encoder\"], None, ck[\"arch\"]).to(self.dev).eval()\n        self.model.load_state_dict(ck[\"model\"])\n\n    def __call__(self, batch: np.ndarray) -> np.ndarray:\n        with self.torch.no_grad(), self.torch.autocast(self.dev.type, enabled=self.dev.type == \"cuda\"):\n            return self.model(self.torch.from_numpy(batch).to(self.dev)).float().cpu().numpy()\n\n\ndef run_scenes(data: Path, ids: list[str], predictor, card: ModelCard, grid: list[dict],\n               suffixes: tuple[str, ...]) -> dict:\n    \"\"\"Probabilities computed ONCE per scene, then scored for every threshold/mask combo.\"\"\"\n    res = {(gi, sfx): {\"cm\": ConfMat(), \"inst\": [0, 0, 0]} for gi in range(len(grid)) for sfx in suffixes}\n    n_scenes = 0\n    for pid in ids:\n        sp = data / \"scenes\" / f\"{pid}.tif\"\n        gts = {sfx: data / \"masks\" / f\"{pid}{sfx}.tif\" for sfx in suffixes}\n        if not sp.exists() or not all(p.exists() for p in gts.values()):\n            continue\n        gts = {sfx: _read_mask(p) for sfx, p in gts.items()}\n        if not any(np.isin(g, config.OIL_CLASSES).any() for g in gts.values()):\n            continue\n        scene = load_scene(sp)\n        probs, _ = predict_scene(scene, predictor, card, batch_size=8)\n        n_scenes += 1\n        for gi, ov in enumerate(grid):\n            feats = extract_slicks(probs, scene.window, scene.valid, overrides=ov)\n            pred = np.zeros(scene.db.shape, np.uint8)\n            if feats:\n                pred = rasterize([(shape(f[\"geometry\"]), f[\"properties\"][\"cls\"]) for f in feats],\n                                 out_shape=pred.shape, transform=scene.window.transform, fill=0,\n                                 dtype=\"uint8\")\n            for sfx, gt in gts.items():\n                if not np.isin(gt, config.OIL_CLASSES).any():\n                    continue\n                r = res[(gi, sfx)]\n                r[\"cm\"].update(pred, gt)\n                tp, fp, fn = instance_scores(feats, gt, scene.window, config.EVAL_INSTANCE_IOU)\n                r[\"inst\"][0] += tp\n                r[\"inst\"][1] += fp\n                r[\"inst\"][2] += fn\n        log.info(\"evaluated %s (%d)\", pid, n_scenes)\n    out = {}\n    for (gi, sfx), r in res.items():\n        s = r[\"cm\"].summary()\n        s[\"instance\"] = _prf(*r[\"inst\"])\n        s[\"thresholds\"] = grid[gi]\n        s[\"n_scenes\"] = n_scenes\n        out[(gi, sfx)] = s\n    return out\n\n\ndef main(argv=None) -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--data\", required=True)\n    ap.add_argument(\"--model\", required=True, help=\"dir with model.onnx + model_card.json\")\n    ap.add_argument(\"--torch-ckpt\", default=None, help=\"use best.pt on GPU instead of ONNX (Colab)\")\n    ap.add_argument(\"--max-val-scenes\", type=int, default=40)\n    args = ap.parse_args(argv)\n    logging.basicConfig(level=logging.INFO, format=\"%(asctime)s %(levelname)s %(message)s\")\n    data, mdir = Path(args.data), Path(args.model)\n    card = ModelCard.load(mdir / \"model_card.json\")\n    predictor = TorchPredictor(Path(args.torch_ckpt)) if args.torch_ckpt else OnnxPredictor(mdir / \"model.onnx\")\n    splits = pd.read_csv(data / \"meta\" / \"splits.csv\")\n\n    grid = [{\"p_seed\": s, \"p_trim\": t} for s in (0.6, 0.75, 0.9) for t in (0.3, 0.5)]\n    val_ids = splits[splits.split == \"val\"].pc_id.tolist()[: args.max_val_scenes]\n    # Val has few human-reviewed scenes, so tune on the full (T1+T2) mask.\n    val = run_scenes(data, val_ids, predictor, card, grid, suffixes=(\"\",))\n    best_i = max(range(len(grid)),\n                 key=lambda i: val[(i, \"\")][\"instance\"][\"f1_dice\"] + val[(i, \"\")][\"OIL_binary\"][\"f1_dice\"])\n    best = grid[best_i]\n    log.info(\"val-tuned thresholds: %s\", best)\n\n    test_ids = splits[splits.split == \"test\"].pc_id.tolist()\n    test = run_scenes(data, test_ids, predictor, card, [best], suffixes=(\"_t1\", \"\"))\n    t1, allm = test[(0, \"_t1\")], test[(0, \"\")]\n    report = {\n        \"model_version\": card.model_version,\n        \"tuned_thresholds\": best,\n        \"val_grid\": {str(grid[i]): {\"oil_pixel_f1\": val[(i, \"\")][\"OIL_binary\"][\"f1_dice\"],\n                                    \"instance_f1\": val[(i, \"\")][\"instance\"][\"f1_dice\"]}\n                     for i in range(len(grid))},\n        \"test_T1_human_reviewed\": t1,\n        \"test_T1plusT2_agreement_with_cerulean\": allm,\n        \"instance_iou_threshold\": config.EVAL_INSTANCE_IOU,\n        \"note\": (\"T1 = human-reviewed Cerulean slicks only (unreviewed = ignore). T1+T2 measures agreement \"\n                 \"with Cerulean's own model output, not ground truth. Cerulean's published pixel F1 0.532 \"\n                 \"was measured on THEIR private test set and is NOT comparable.\"),\n    }\n    (mdir / \"test_metrics.json\").write_text(json.dumps(report, indent=2, default=float))\n    log.info(\"TEST T1 (%d scenes): oil pixel P=%.3f R=%.3f F1=%.3f IoU=%.3f | instance P=%.3f R=%.3f F1=%.3f\",\n             t1[\"n_scenes\"], t1[\"OIL_binary\"][\"precision\"], t1[\"OIL_binary\"][\"recall\"],\n             t1[\"OIL_binary\"][\"f1_dice\"], t1[\"OIL_binary\"][\"iou\"], t1[\"instance\"][\"precision\"],\n             t1[\"instance\"][\"recall\"], t1[\"instance\"][\"f1_dice\"])\n    raw = card.raw\n    raw[\"thresholds\"].update(best)\n    raw[\"metrics\"] = {\"test_T1_oil_pixel\": t1[\"OIL_binary\"], \"test_T1_instance\": t1[\"instance\"],\n                      \"test_T1plusT2_oil_pixel\": allm[\"OIL_binary\"],\n                      \"test_T1plusT2_instance\": allm[\"instance\"],\n                      \"per_class_T1\": {k: t1[k] for k in (\"INFRA\", \"NATURAL\", \"VESSEL\")}}\n    (mdir / \"model_card.json\").write_text(json.dumps(raw, indent=2, default=float))\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "pyproject.toml": "[build-system]\nrequires = [\"setuptools>=68\", \"wheel\"]\nbuild-backend = \"setuptools.build_meta\"\n\n[project]\nname = \"oilspill\"\nversion = \"0.1.0\"\ndescription = \"Open-source reproduction of SkyTruth Cerulean's Sentinel-1 oil-slick detection pipeline\"\nreadme = \"README.md\"\nrequires-python = \">=3.12\"\nlicense = { text = \"Apache-2.0\" }\nauthors = [{ name = \"oilspill contributors\" }]\n\n# Versions here are DELIBERATELY loose until Phase 0 records what actually\n# resolved on this machine; see requirements.lock.txt for the pinned set.\ndependencies = [\n  \"numpy\",\n  \"pandas\",\n  \"pyarrow\",\n  \"rasterio\",\n  \"shapely>=2.0\",\n  # NOT pyproj / geopandas / pyogrio: Windows Smart App Control blocks their\n  # native extensions on the dev machine (DECISIONS.md D0.6). All CRS work goes\n  # through rasterio's bundled GDAL/PROJ (oilspill.crsutil) and shapefiles are\n  # read with pure-Python pyshp (oilspill.landmask).\n  \"pyshp\",\n  \"pystac-client\",\n  \"planetary-computer\",\n  \"httpx\",\n  \"tenacity\",\n  \"Pillow\",\n  \"matplotlib\",\n  \"tqdm\",\n  \"python-dotenv\",\n  \"scipy\",\n]\n\n[project.optional-dependencies]\ntrain = [\n  \"torch\",\n  \"torchvision\",\n  \"segmentation-models-pytorch==0.5.0\",\n  \"timm\",\n  \"albumentations\",\n  \"tensorboard\",\n  \"scikit-learn\",\n]\nexport = [\"onnx\", \"onnxruntime\"]\napi = [\n  \"fastapi\",\n  \"uvicorn[standard]\",\n  \"python-multipart\",\n  \"sqlalchemy>=2\",\n  \"geoalchemy2\",\n  \"alembic\",\n]\ndev = [\"pytest\", \"pytest-cov\", \"ruff\"]\n\n[tool.setuptools.packages.find]\nwhere = [\"src\"]\n\n[tool.pytest.ini_options]\ntestpaths = [\"tests\"]\nfilterwarnings = [\"ignore::DeprecationWarning\"]\n\n[tool.ruff]\nline-length = 100\ntarget-version = \"py312\"\n\n[tool.ruff.lint]\nselect = [\"E\", \"F\", \"I\", \"W\", \"UP\", \"B\"]\nignore = [\"E501\"]\n", "README.md": "oilspill (Colab copy)\n"}

# 2) write the package onto the runtime and install it
import pathlib, subprocess, sys
PKG_ROOT = pathlib.Path('/content/oilspill_pkg')
for rel, text in FILES.items():
    p = PKG_ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'segmentation-models-pytorch==0.5.0', 'timm', 'rasterio', 'pystac-client',
                'planetary-computer', 'shapely>=2', 'pyshp', 'tenacity', 'python-dotenv',
                'onnx', 'onnxruntime', 'httpx', 'pyarrow'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(PKG_ROOT)], check=True)
import importlib, site; importlib.invalidate_caches()
sys.path.insert(0, str(PKG_ROOT / 'src'))
import oilspill, oilspill.config as C

def run_live(args):
    # run a step as a subprocess and stream its log into this notebook live
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1, cwd='/content')
    for line in p.stdout:
        if 'HTTP Request:' in line:
            continue  # too chatty
        print(line, end='', flush=True)
    return p.wait()

print('oilspill installed; grid pixel =', C.PIXEL_DEG, 'deg')

In [ ]:
# 3) GPU check
import torch, subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout[:800])
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> T4 GPU, then Run All again.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 4) Build the dataset (real labels + real imagery). ~20-40 min.
# Built DIRECTLY on Google Drive, one status file per scene, so a Colab disconnect
# loses nothing: re-run and it continues from the last finished scene.
import shutil, subprocess, sys, json
DATA_DRIVE = DRIVE / 'dataset'
OLD_TAR = DRIVE / 'dataset.tar'   # written by the first version of this notebook
if not (DATA / 'meta' / 'dataset_meta.json').exists() and OLD_TAR.exists():
    import tarfile
    print('restoring dataset from', OLD_TAR, '...')
    DATA.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(OLD_TAR) as t: t.extractall(DATA.parent)
if not (DATA / 'meta' / 'dataset_meta.json').exists() and not (DATA_DRIVE / 'meta' / 'dataset_meta.json').exists():
    r = run_live([sys.executable, '-m', 'oilspill.dataset_build', '--root', str(DATA_DRIVE),
                        '--scenes-per-aoi', '40', '--workers', '8'])
    assert r == 0, 'dataset build failed - scroll up for the error'
# copy to the fast local disk for training (Drive reads are slow)
if not (DATA / 'meta' / 'dataset_meta.json').exists():
    print('copying dataset Drive -> /content (fast local disk) ...')
    shutil.copytree(DATA_DRIVE, DATA, dirs_exist_ok=True, ignore=shutil.ignore_patterns('cache'))
meta = json.loads((DATA / 'meta' / 'dataset_meta.json').read_text())
print(json.dumps({k: meta[k] for k in ['n_done','n_failed','db_clip','split_counts','t1_scenes_by_split','pos_px_total']}, indent=1))

In [ ]:
# 5) Look at the data: SAR + label overlay for random training scenes
import numpy as np, pandas as pd, rasterio, matplotlib.pyplot as plt
sp = pd.read_csv(DATA / 'meta' / 'splits.csv')
cands = sp[(sp.split == 'train')].sample(min(4, (sp.split=='train').sum()), random_state=0)
fig, axes = plt.subplots(len(cands), 2, figsize=(12, 5 * len(cands)))
axes = np.atleast_2d(axes)
for ax, pid in zip(axes, cands.pc_id):
    with rasterio.open(DATA / 'scenes' / f'{pid}.tif') as d: db = d.read(1) / 100.0
    with rasterio.open(DATA / 'masks' / f'{pid}.tif') as d: m = d.read(1)
    lo, hi = meta['db_clip']
    ax[0].imshow(np.clip((db - lo) / (hi - lo), 0, 1), cmap='gray'); ax[0].set_title(pid[:40], fontsize=8)
    ax[1].imshow(np.clip((db - lo) / (hi - lo), 0, 1), cmap='gray')
    ov = np.ma.masked_where(~np.isin(m, [1, 2, 3]), m)
    ax[1].imshow(ov, cmap='autumn', alpha=0.8, vmin=1, vmax=3); ax[1].set_title('labels (1 infra, 2 natural, 3 vessel)')
    for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 6) Sanity gate: the model must be able to overfit 16 crops (catches data/label/loss bugs)
import subprocess, sys
r = run_live([sys.executable, '-m', 'oilspill.train', '--data', str(DATA), '--out', str(DRIVE / 'overfit'),
                    '--overfit', '16', '--epochs', '150', '--batch-size', '8', '--workers', '0', '--lr', '1e-3'])
print('overfit gate exit code', r, '(0 = passed, 2 = failed)')
assert r == 0, 'Overfit gate failed - paste the log above to Claude'

In [ ]:
# 7) Full training (resumable: just re-run this cell after a disconnect)
import subprocess, sys
r = run_live([sys.executable, '-m', 'oilspill.train', '--data', str(DATA), '--out', str(RUN),
                    '--epochs', '40', '--samples-per-epoch', '2400', '--batch-size', '8',
                    '--workers', '2', '--resume', '--max-hours', '9'])
assert r == 0

In [ ]:
# 8) Training curves
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(RUN / 'log.csv')
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
log.plot(x='epoch', y='loss', ax=ax[0], title='train loss')
log.plot(x='epoch', y=['val_oil_dice', 'val_infra_f1', 'val_natural_f1', 'val_vessel_f1'], ax=ax[1], title='val (fixed crops)')
plt.show(); print(log.tail(5).to_string())

In [ ]:
# 9) Export ONNX (+ model_card.json) and verify ONNX == torch
import subprocess, sys
r = run_live([sys.executable, '-m', 'oilspill.export', '--run', str(RUN), '--out', str(RUN / 'export')])
assert r == 0

In [ ]:
# 10) Honest evaluation: thresholds tuned on VAL, TEST scored once on human-reviewed slicks
import subprocess, sys, json
r = run_live([sys.executable, '-m', 'oilspill.evaluate', '--data', str(DATA), '--model', str(RUN / 'export'),
                    '--torch-ckpt', str(RUN / 'best.pt')])
assert r == 0
rep = json.loads((RUN / 'export' / 'test_metrics.json').read_text())
t = rep['test_T1_human_reviewed']
print('tuned thresholds:', rep['tuned_thresholds'])
print('TEST (human-reviewed): oil pixel', {k: round(v, 3) for k, v in t['OIL_binary'].items() if k in ('precision','recall','f1_dice','iou')})
print('TEST (human-reviewed): instance', {k: round(v, 3) for k, v in t['instance'].items() if k in ('precision','recall','f1_dice')})
print('Cerulean published pixel F1 0.532 is on THEIR private test set - not directly comparable.')

In [ ]:
# 11) Visual check: full-scene prediction on a test scene with our polygons vs Cerulean labels
import numpy as np, matplotlib.pyplot as plt, rasterio, pandas as pd
from shapely.geometry import shape
from oilspill.infer import ModelCard, OnnxPredictor, predict_scene
from oilspill.postprocess import extract_slicks
from oilspill.scene import load_scene
from oilspill.evaluate import TorchPredictor
card = ModelCard.load(RUN / 'export' / 'model_card.json')
pred = TorchPredictor(RUN / 'best.pt')
sp = pd.read_csv(DATA / 'meta' / 'splits.csv')
pid = sp[(sp.split == 'test') & (sp.t1_pos_px > 0)].pc_id.iloc[0]
scene = load_scene(DATA / 'scenes' / f'{pid}.tif')
probs, timing = predict_scene(scene, pred, card, batch_size=8)
feats = extract_slicks(probs, scene.window, scene.valid, overrides=card.thresholds)
with rasterio.open(DATA / 'masks' / f'{pid}.tif') as d: m = d.read(1)
lo, hi = card.db_clip
b = scene.bounds; ext = [b[0], b[2], b[1], b[3]]
fig, ax = plt.subplots(1, 2, figsize=(16, 8))
for a in ax: a.imshow(np.clip((scene.db - lo) / (hi - lo), 0, 1), cmap='gray', extent=ext)
ax[0].imshow(np.ma.masked_where(~np.isin(m, [1,2,3]), m), cmap='autumn', alpha=.7, extent=ext); ax[0].set_title('Cerulean labels')
ax[1].imshow(np.ma.masked_less(probs[1:].sum(0), 0.3), cmap='magma', alpha=.6, extent=ext, vmin=0, vmax=1)
for f in feats:
    g = shape(f['geometry'])
    for p in g.geoms: ax[1].plot(*p.exterior.xy, color='cyan', lw=1)
ax[1].set_title(f'our model: {len(feats)} slicks'); plt.show()
print(timing); [print(f['properties']['cls_name'], round(f['properties']['confidence'], 3), round(f['properties']['area_km2'], 2), 'km2') for f in feats[:10]]

In [ ]:
# 12) Package the model for the laptop: MyDrive/oilspill/oilspill_model.zip
import zipfile, json
z = DRIVE / 'oilspill_model.zip'
with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
    for n in ['model.onnx', 'model_card.json', 'test_metrics.json']:
        zf.write(RUN / 'export' / n, n)
    zf.write(RUN / 'log.csv', 'train_log.csv')
print('WROTE', z, round(z.stat().st_size / 1e6, 1), 'MB')
print(json.dumps(json.loads((RUN / 'export' / 'model_card.json').read_text())['metrics'], indent=1)[:1500])
print()
print('NEXT: open drive.google.com -> My Drive -> oilspill -> download oilspill_model.zip,')
print('      then unzip it into the project folder  sih\\models\\')